In [ ]:
!pip -q install kagglehub

In [ ]:
import os
import json
from collections import Counter

import kagglehub
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
import os
import json
from typing import List

import kagglehub
import pandas as pd

# -------------------------
# Config
# -------------------------
TARGET_CITIES = ["Philadelphia"]   # add more later if needed
KEEP_ONLY_OPEN = True
OUTPUT_DIR = "/content/yelp_philly_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download dataset
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
print("Dataset path:", path)

BUSINESS_FILE = os.path.join(path, "yelp_academic_dataset_business.json")
REVIEW_FILE   = os.path.join(path, "yelp_academic_dataset_review.json")
USER_FILE     = os.path.join(path, "yelp_academic_dataset_user.json")

100%|██████████| 4.07G/4.07G [01:00<00:00, 72.6MB/s]

Extracting files...


In [ ]:
def normalize_city(city: str) -> str:
    return str(city).strip().lower()


TARGET_CITY_SET = {normalize_city(city) for city in TARGET_CITIES}


def is_restaurant_business(categories: str) -> bool:
    if not categories:
        return False
    category_list = [c.strip() for c in str(categories).split(",")]
    return "Restaurants" in category_list


def safe_json_dumps(x):
    if x is None:
        return None
    if isinstance(x, (dict, list)):
        return json.dumps(x)
    return str(x)

In [ ]:
def preprocess_businesses(filepath: str) -> pd.DataFrame:
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)

            city = normalize_city(row.get("city", ""))
            if city not in TARGET_CITY_SET:
                continue

            if KEEP_ONLY_OPEN and row.get("is_open", 0) != 1:
                continue

            if not row.get("business_id") or not row.get("name"):
                continue

            categories = row.get("categories")
            if not is_restaurant_business(categories):
                continue

            rows.append({
                "business_id": row["business_id"],
                "name": row["name"],
                "address": row.get("address"),
                "city": row.get("city"),
                "state": row.get("state"),
                "postal_code": row.get("postal_code"),
                "latitude": row.get("latitude"),
                "longitude": row.get("longitude"),
                "stars": row.get("stars"),
                "review_count": row.get("review_count"),
                "is_open": row.get("is_open"),
                "categories": row.get("categories"),         # keep raw for later
                "attributes_json": safe_json_dumps(row.get("attributes")),
                "hours_json": safe_json_dumps(row.get("hours"))
            })

    df = pd.DataFrame(rows)
    return df

In [ ]:
def preprocess_reviews(filepath: str, valid_business_ids: set) -> pd.DataFrame:
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            row = json.loads(line)

            if row.get("business_id") not in valid_business_ids:
                continue

            if not row.get("review_id") or not row.get("user_id") or not row.get("text"):
                continue

            dt = pd.to_datetime(row.get("date"), errors="coerce")
            if pd.isna(dt):
                continue

            rows.append({
                "review_id": row["review_id"],
                "user_id": row["user_id"],
                "business_id": row["business_id"],
                "stars": row.get("stars"),
                "useful": row.get("useful"),
                "funny": row.get("funny"),
                "cool": row.get("cool"),
                "text": row.get("text"),
                "date": dt.strftime("%Y-%m-%d"),
                "review_year": dt.year,
                "review_month": dt.month
            })

            if (i + 1) % 500000 == 0:
                print(f"Processed {i+1:,} review lines... kept {len(rows):,}")

    df = pd.DataFrame(rows)
    return df

In [ ]:
def preprocess_users(filepath: str, valid_user_ids: set) -> pd.DataFrame:
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)

            if row.get("user_id") not in valid_user_ids:
                continue

            if not row.get("user_id"):
                continue

            yelping_since = pd.to_datetime(row.get("yelping_since"), errors="coerce")
            yelping_year = None if pd.isna(yelping_since) else yelping_since.year

            rows.append({
                "user_id": row["user_id"],
                "name": row.get("name"),
                "review_count": row.get("review_count"),
                "average_stars": row.get("average_stars"),
                "fans": row.get("fans"),
                "useful": row.get("useful"),
                "funny": row.get("funny"),
                "cool": row.get("cool"),
                "yelping_since": None if pd.isna(yelping_since) else yelping_since.strftime("%Y-%m-%d"),
                "yelping_year": yelping_year,
                "elite": safe_json_dumps(row.get("elite")),
                "friends": safe_json_dumps(row.get("friends"))
            })

    df = pd.DataFrame(rows)
    return df

In [ ]:
print("Processing businesses...")
businesses = preprocess_businesses(BUSINESS_FILE)
valid_business_ids = set(businesses["business_id"])
print("Businesses kept:", len(businesses))

print("\nProcessing reviews...")
reviews = preprocess_reviews(REVIEW_FILE, valid_business_ids)
valid_user_ids = set(reviews["user_id"])
print("Reviews kept:", len(reviews))

print("\nProcessing users...")
users = preprocess_users(USER_FILE, valid_user_ids)
print("Users kept:", len(users))

In [ ]:
businesses.to_csv(os.path.join(OUTPUT_DIR, "businesses.csv"), index=False)
reviews.to_csv(os.path.join(OUTPUT_DIR, "reviews.csv"), index=False)
users.to_csv(os.path.join(OUTPUT_DIR, "users.csv"), index=False)

print("\nSaved files:")
print("-", os.path.join(OUTPUT_DIR, "businesses.csv"))
print("-", os.path.join(OUTPUT_DIR, "reviews.csv"))
print("-", os.path.join(OUTPUT_DIR, "users.csv"))

In [ ]:
print("businesses shape:", businesses.shape)
print("reviews shape:", reviews.shape)
print("users shape:", users.shape)

print("\nUnique cities kept:", businesses["city"].nunique())
print("Restaurants with at least 1 kept review:", reviews["business_id"].nunique())
print("Unique users with kept reviews:", reviews["user_id"].nunique())

display(businesses.head())
display(reviews.head())
display(users.head())

In [ ]:
# =========================================================
# Config
# =========================================================
DATA_DIR = "/content/yelp_philly_processed"

BUSINESSES_PATH = os.path.join(DATA_DIR, "businesses.csv")
REVIEWS_PATH = os.path.join(DATA_DIR, "reviews.csv")
USERS_PATH = os.path.join(DATA_DIR, "users.csv")

OUTPUT_DIR = os.path.join(DATA_DIR, "splits_temporal")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# temporal split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# define cold-start from TRAIN only
COLD_START_MAX_TRAIN_REVIEWS = 5

# main benchmark:
# False = keep only items seen at least once in train for val/test
# True  = allow zero-train items in val/test
ALLOW_ZERO_TRAIN_ITEMS_IN_EVAL = False

# optional user-history filter
# set to 0 to disable
MIN_TRAIN_USER_INTERACTIONS = 1

# optional save of summary csvs
SAVE_SUMMARY_TABLES = True


# =========================================================
# Load processed files
# =========================================================
businesses = pd.read_csv(BUSINESSES_PATH)
reviews = pd.read_csv(REVIEWS_PATH)
users = pd.read_csv(USERS_PATH)

print("Loaded:")
print("businesses:", businesses.shape)
print("reviews:", reviews.shape)
print("users:", users.shape)


# =========================================================
# Clean and sort reviews by time
# =========================================================
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")
reviews = reviews.dropna(subset=["date", "user_id", "business_id"]).copy()
reviews = reviews.sort_values("date").reset_index(drop=True)

print("\nReviews after cleanup:", reviews.shape)
print("Date range:", reviews["date"].min(), "to", reviews["date"].max())


# =========================================================
# Global temporal split
# =========================================================
train_cutoff = reviews["date"].quantile(TRAIN_RATIO)
val_cutoff = reviews["date"].quantile(TRAIN_RATIO + VAL_RATIO)

train_reviews = reviews[reviews["date"] <= train_cutoff].copy()
val_reviews = reviews[(reviews["date"] > train_cutoff) & (reviews["date"] <= val_cutoff)].copy()
test_reviews = reviews[reviews["date"] > val_cutoff].copy()

print("\nInitial time split:")
print("train:", train_reviews.shape)
print("val:", val_reviews.shape)
print("test:", test_reviews.shape)
print("train cutoff:", train_cutoff)
print("val cutoff:", val_cutoff)


# =========================================================
# Keep only users seen in train
# =========================================================
train_user_ids = set(train_reviews["user_id"].unique())

val_reviews = val_reviews[val_reviews["user_id"].isin(train_user_ids)].copy()
test_reviews = test_reviews[test_reviews["user_id"].isin(train_user_ids)].copy()

print("\nAfter keeping only users seen in train:")
print("train:", train_reviews.shape)
print("val:", val_reviews.shape)
print("test:", test_reviews.shape)


# =========================================================
# Optional: require minimum train interactions per user
# =========================================================
if MIN_TRAIN_USER_INTERACTIONS > 1:
    train_user_counts = (
        train_reviews.groupby("user_id")
        .size()
        .reset_index(name="train_user_review_count")
    )

    eligible_train_users = set(
        train_user_counts.loc[
            train_user_counts["train_user_review_count"] >= MIN_TRAIN_USER_INTERACTIONS,
            "user_id"
        ]
    )

    train_reviews = train_reviews[train_reviews["user_id"].isin(eligible_train_users)].copy()
    val_reviews = val_reviews[val_reviews["user_id"].isin(eligible_train_users)].copy()
    test_reviews = test_reviews[test_reviews["user_id"].isin(eligible_train_users)].copy()

    print(f"\nAfter enforcing MIN_TRAIN_USER_INTERACTIONS >= {MIN_TRAIN_USER_INTERACTIONS}:")
    print("train:", train_reviews.shape)
    print("val:", val_reviews.shape)
    print("test:", test_reviews.shape)


# =========================================================
# Compute TRAIN restaurant counts
# =========================================================
train_business_counts = (
    train_reviews.groupby("business_id")
    .size()
    .reset_index(name="train_review_count")
)

businesses_with_train_info = businesses.merge(
    train_business_counts,
    on="business_id",
    how="left"
)

businesses_with_train_info["train_review_count"] = (
    businesses_with_train_info["train_review_count"]
    .fillna(0)
    .astype(int)
)

# cold-start = 1..threshold reviews in train
businesses_with_train_info["cold_start_in_train"] = (
    (businesses_with_train_info["train_review_count"] >= 1) &
    (businesses_with_train_info["train_review_count"] <= COLD_START_MAX_TRAIN_REVIEWS)
)

# zero-train bucket
businesses_with_train_info["zero_train_reviews"] = (
    businesses_with_train_info["train_review_count"] == 0
)

seen_train_business_ids = set(
    businesses_with_train_info.loc[
        businesses_with_train_info["train_review_count"] >= 1,
        "business_id"
    ]
)

cold_start_business_ids = set(
    businesses_with_train_info.loc[
        businesses_with_train_info["cold_start_in_train"],
        "business_id"
    ]
)

zero_train_business_ids = set(
    businesses_with_train_info.loc[
        businesses_with_train_info["zero_train_reviews"],
        "business_id"
    ]
)

print("\nTrain business summary:")
print("businesses seen in train:", len(seen_train_business_ids))
print(f"cold-start businesses (1 to {COLD_START_MAX_TRAIN_REVIEWS} train reviews):", len(cold_start_business_ids))
print("zero-train businesses:", len(zero_train_business_ids))


# =========================================================
# Main benchmark option: remove zero-train items from val/test
# =========================================================
if not ALLOW_ZERO_TRAIN_ITEMS_IN_EVAL:
    val_reviews = val_reviews[val_reviews["business_id"].isin(seen_train_business_ids)].copy()
    test_reviews = test_reviews[test_reviews["business_id"].isin(seen_train_business_ids)].copy()

    print("\nAfter dropping zero-train items from val/test:")
    print("train:", train_reviews.shape)
    print("val:", val_reviews.shape)
    print("test:", test_reviews.shape)


# =========================================================
# Add cold-start labels to each split
# =========================================================
train_reviews["cold_start_in_train"] = train_reviews["business_id"].isin(cold_start_business_ids)
val_reviews["cold_start_in_train"] = val_reviews["business_id"].isin(cold_start_business_ids)
test_reviews["cold_start_in_train"] = test_reviews["business_id"].isin(cold_start_business_ids)

train_reviews["zero_train_reviews"] = False
val_reviews["zero_train_reviews"] = val_reviews["business_id"].isin(zero_train_business_ids)
test_reviews["zero_train_reviews"] = test_reviews["business_id"].isin(zero_train_business_ids)


# =========================================================
# Helper tables
# =========================================================
train_users = pd.DataFrame({"user_id": sorted(train_reviews["user_id"].unique())})
train_items = pd.DataFrame({"business_id": sorted(train_reviews["business_id"].unique())})

cold_start_items = businesses_with_train_info[
    businesses_with_train_info["cold_start_in_train"]
].copy()

warm_items = businesses_with_train_info[
    businesses_with_train_info["train_review_count"] > COLD_START_MAX_TRAIN_REVIEWS
].copy()

zero_train_items = businesses_with_train_info[
    businesses_with_train_info["zero_train_reviews"]
].copy()


# =========================================================
# Summary helpers
# =========================================================
def summarize_split(name, df):
    return {
        "split": name,
        "rows": len(df),
        "unique_users": df["user_id"].nunique(),
        "unique_businesses": df["business_id"].nunique(),
        "date_min": df["date"].min(),
        "date_max": df["date"].max(),
        "cold_start_rows": int(df["cold_start_in_train"].sum()),
        "cold_start_businesses": df.loc[df["cold_start_in_train"], "business_id"].nunique(),
        "zero_train_rows": int(df["zero_train_reviews"].sum()),
    }


train_summary = summarize_split("train", train_reviews)
val_summary = summarize_split("val", val_reviews)
test_summary = summarize_split("test", test_reviews)

summary_df = pd.DataFrame([train_summary, val_summary, test_summary])

print("\nSplit summary:")
print(summary_df)


# =========================================================
# Business summary table
# =========================================================
business_summary_df = pd.DataFrame({
    "metric": [
        "total_businesses",
        "businesses_seen_in_train",
        f"cold_start_businesses_1_to_{COLD_START_MAX_TRAIN_REVIEWS}",
        "zero_train_businesses",
        "median_train_review_count_seen_items",
        "mean_train_review_count_seen_items"
    ],
    "value": [
        len(businesses_with_train_info),
        len(seen_train_business_ids),
        len(cold_start_business_ids),
        len(zero_train_business_ids),
        businesses_with_train_info.loc[
            businesses_with_train_info["train_review_count"] >= 1,
            "train_review_count"
        ].median(),
        businesses_with_train_info.loc[
            businesses_with_train_info["train_review_count"] >= 1,
            "train_review_count"
        ].mean(),
    ]
})

print("\nBusiness summary:")
print(business_summary_df)


# =========================================================
# Save outputs
# =========================================================
train_reviews.to_csv(os.path.join(OUTPUT_DIR, "train_reviews.csv"), index=False)
val_reviews.to_csv(os.path.join(OUTPUT_DIR, "val_reviews.csv"), index=False)
test_reviews.to_csv(os.path.join(OUTPUT_DIR, "test_reviews.csv"), index=False)

businesses_with_train_info.to_csv(
    os.path.join(OUTPUT_DIR, "businesses_with_train_info.csv"),
    index=False
)

train_users.to_csv(os.path.join(OUTPUT_DIR, "train_users.csv"), index=False)
train_items.to_csv(os.path.join(OUTPUT_DIR, "train_items.csv"), index=False)
cold_start_items.to_csv(os.path.join(OUTPUT_DIR, "cold_start_items.csv"), index=False)
warm_items.to_csv(os.path.join(OUTPUT_DIR, "warm_items.csv"), index=False)
zero_train_items.to_csv(os.path.join(OUTPUT_DIR, "zero_train_items.csv"), index=False)

if SAVE_SUMMARY_TABLES:
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "split_summary.csv"), index=False)
    business_summary_df.to_csv(os.path.join(OUTPUT_DIR, "business_summary.csv"), index=False)

print("\nSaved files:")
for fname in [
    "train_reviews.csv",
    "val_reviews.csv",
    "test_reviews.csv",
    "businesses_with_train_info.csv",
    "train_users.csv",
    "train_items.csv",
    "cold_start_items.csv",
    "warm_items.csv",
    "zero_train_items.csv",
    "split_summary.csv",
    "business_summary.csv",
]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        print("-", fpath)


# =========================================================
# Quick previews
# =========================================================
print("\nTrain preview:")
display(train_reviews.head())

print("\nVal preview:")
display(val_reviews.head())

print("\nTest preview:")
display(test_reviews.head())

print("\nBusiness preview:")
display(
    businesses_with_train_info[
        ["business_id", "name", "review_count", "train_review_count", "cold_start_in_train", "zero_train_reviews"]
    ].head()
)

print("\nSplit summary preview:")
display(summary_df)

print("\nBusiness summary preview:")
display(business_summary_df)

## EVALUATOR CELL use this to evaluate all models

In [ ]:
# =========================================================
# Shared evaluator for all recommendation models
# =========================================================
import math
import numpy as np

MIN_POSITIVE_STARS = 4.0
EVAL_KS = [5, 10, 20]

# If this cell is run later by itself, reload the split files.
if "train_reviews" not in globals():
    SPLIT_DIR = os.path.join(DATA_DIR, "splits_temporal")
    train_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "train_reviews.csv"))
    val_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "val_reviews.csv"))
    test_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "test_reviews.csv"))


def make_seen_map(train_df):
    seen = {}
    for uid, grp in train_df.groupby("user_id"):
        seen[uid] = set(grp["business_id"])
    return seen


def make_truth_map(split_df, min_stars=MIN_POSITIVE_STARS, cold_only=False):
    part = split_df[split_df["stars"] >= min_stars].copy()

    if cold_only:
        part = part[part["cold_start_in_train"] == True].copy()

    truth = {}
    for uid, grp in part.groupby("user_id"):
        truth[uid] = set(grp["business_id"])
    return truth


def recommendation_df_to_map(rec_df, topk=20):
    """Accept either rank_1...rank_k format or user_id/business_id/score format."""
    rank_cols = [c for c in rec_df.columns if c.startswith("rank_")]

    rec_map = {}

    if rank_cols:
        rank_cols = sorted(rank_cols, key=lambda x: int(x.split("_")[1]))
        for row in rec_df[["user_id"] + rank_cols].itertuples(index=False):
            uid = row[0]
            rec_map[uid] = [x for x in row[1:] if pd.notna(x)][:topk]
        return rec_map

    if "score" in rec_df.columns:
        rec_df = rec_df.sort_values(["user_id", "score"], ascending=[True, False])

    for uid, grp in rec_df.groupby("user_id"):
        rec_map[uid] = list(grp["business_id"].head(topk))

    return rec_map


def clean_rec_list(items, seen_items=None, k=10):
    seen_items = seen_items or set()
    out = []
    used = set()

    for item in items:
        if item in used:
            continue
        if item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == k:
            break

    return out


def evaluate_rec_map(rec_map, truth_map, seen_map=None, k=10):
    rows = []

    for uid, true_items in truth_map.items():
        if len(true_items) == 0:
            continue

        seen_items = set() if seen_map is None else seen_map.get(uid, set())
        recs = clean_rec_list(rec_map.get(uid, []), seen_items=seen_items, k=k)

        hit_flags = [1 if item in true_items else 0 for item in recs]
        hits = sum(hit_flags)

        hit_at_k = 1 if hits > 0 else 0
        recall_at_k = hits / len(true_items)

        rr = 0.0
        for rank, hit in enumerate(hit_flags, start=1):
            if hit:
                rr = 1.0 / rank
                break

        dcg = 0.0
        for rank, hit in enumerate(hit_flags, start=1):
            if hit:
                dcg += 1.0 / math.log2(rank + 1)

        ideal_hits = min(len(true_items), k)
        idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
        ndcg = 0.0 if idcg == 0 else dcg / idcg

        rows.append({
            "hit": hit_at_k,
            "recall": recall_at_k,
            "mrr": rr,
            "ndcg": ndcg,
        })

    if not rows:
        return {
            "users_eval": 0,
            "hit@k": 0.0,
            "recall@k": 0.0,
            "mrr@k": 0.0,
            "ndcg@k": 0.0,
        }

    out = pd.DataFrame(rows)
    return {
        "users_eval": len(out),
        "hit@k": out["hit"].mean(),
        "recall@k": out["recall"].mean(),
        "mrr@k": out["mrr"].mean(),
        "ndcg@k": out["ndcg"].mean(),
    }


def evaluate_model(model_name, recs, split_df, split_name="val", k_list=EVAL_KS):
    rec_map = recs if isinstance(recs, dict) else recommendation_df_to_map(recs, topk=max(k_list))
    seen_map = make_seen_map(train_reviews)

    all_truth = make_truth_map(split_df, cold_only=False)
    cold_truth = make_truth_map(split_df, cold_only=True)

    eval_rows = []
    for k in k_list:
        for group_name, truth_map in [("all_positive", all_truth), ("cold_start_positive", cold_truth)]:
            scores = evaluate_rec_map(rec_map, truth_map, seen_map=seen_map, k=k)
            eval_rows.append({
                "model": model_name,
                "split": split_name,
                "group": group_name,
                "k": k,
                **scores,
            })

    return pd.DataFrame(eval_rows)


print("Evaluator ready")
print("positive review cutoff:", MIN_POSITIVE_STARS)
print("val positive users:", len(make_truth_map(val_reviews)))
print("val cold-start positive users:", len(make_truth_map(val_reviews, cold_only=True)))
print("test positive users:", len(make_truth_map(test_reviews)))
print("test cold-start positive users:", len(make_truth_map(test_reviews, cold_only=True)))


/tmp/ipykernel_17670/2258142244.py:13: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  train_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "train_reviews.csv"))


Evaluator ready
positive review cutoff: 4.0
val positive users: 7551
val cold-start positive users: 363
test positive users: 5003
test cold-start positive users: 183


In [ ]:
# # =========================================================
# # Small evaluator smoke test: popularity baseline
# # =========================================================
# # This is not the final model. It just proves the evaluator works.

# train_pop_items = (
#     train_reviews.groupby("business_id")
#     .size()
#     .sort_values(ascending=False)
#     .index
#     .tolist()
# )

# val_users = sorted(val_reviews["user_id"].unique())
# pop_recs = {uid: train_pop_items[:100] for uid in val_users}

# pop_val_scores = evaluate_model(
#     "popularity_smoke_test",
#     pop_recs,
#     val_reviews,
#     split_name="val",
#     k_list=[5, 10, 20]
# )

# display(pop_val_scores)


## RERUN PROCESSED DATA so you dont have to keep running validation pipeline

In [ ]:
# =========================================================
# Fast reload saved processed data
# =========================================================
# Run this cell when the processed CSVs already exist and you do not
# want to rerun the JSON cleaning + temporal split pipeline.

import os
import pandas as pd

DATA_DIR = globals().get("DATA_DIR", "/content/yelp_philly_processed")
if not os.path.exists(DATA_DIR) and os.path.exists("yelp_philly_processed"):
    DATA_DIR = "yelp_philly_processed"

SPLIT_DIR = os.path.join(DATA_DIR, "splits_temporal")
RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

BUSINESSES_PATH = os.path.join(DATA_DIR, "businesses.csv")
REVIEWS_PATH = os.path.join(DATA_DIR, "reviews.csv")
USERS_PATH = os.path.join(DATA_DIR, "users.csv")

# Same settings used when the split files were created.
COLD_START_MAX_TRAIN_REVIEWS = 5
ALLOW_ZERO_TRAIN_ITEMS_IN_EVAL = False
MIN_POSITIVE_STARS = globals().get("MIN_POSITIVE_STARS", 4.0)
EVAL_KS = globals().get("EVAL_KS", [5, 10, 20])


def read_saved_csv(path):
    df = pd.read_csv(path)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df


def read_optional_csv(path):
    if os.path.exists(path):
        return read_saved_csv(path)
    return pd.DataFrame()


businesses = read_saved_csv(BUSINESSES_PATH)
reviews = read_saved_csv(REVIEWS_PATH)
users = read_saved_csv(USERS_PATH)

train_reviews = read_saved_csv(os.path.join(SPLIT_DIR, "train_reviews.csv"))
val_reviews = read_saved_csv(os.path.join(SPLIT_DIR, "val_reviews.csv"))
test_reviews = read_saved_csv(os.path.join(SPLIT_DIR, "test_reviews.csv"))

businesses_with_train_info = read_saved_csv(os.path.join(SPLIT_DIR, "businesses_with_train_info.csv"))
train_users = read_saved_csv(os.path.join(SPLIT_DIR, "train_users.csv"))
train_items = read_saved_csv(os.path.join(SPLIT_DIR, "train_items.csv"))
cold_start_items = read_saved_csv(os.path.join(SPLIT_DIR, "cold_start_items.csv"))
warm_items = read_saved_csv(os.path.join(SPLIT_DIR, "warm_items.csv"))
zero_train_items = read_saved_csv(os.path.join(SPLIT_DIR, "zero_train_items.csv"))

summary_df = read_optional_csv(os.path.join(SPLIT_DIR, "split_summary.csv"))
business_summary_df = read_optional_csv(os.path.join(SPLIT_DIR, "business_summary.csv"))

valid_business_ids = set(businesses["business_id"])
valid_user_ids = set(users["user_id"])
seen_train_business_ids = set(train_items["business_id"])
cold_start_business_ids = set(cold_start_items["business_id"])
warm_business_ids = set(warm_items["business_id"])
zero_train_business_ids = set(zero_train_items["business_id"])

print("Saved data loaded")
print("DATA_DIR:", DATA_DIR)
print("businesses:", businesses.shape)
print("reviews:", reviews.shape)
print("users:", users.shape)
print("train/val/test:", train_reviews.shape, val_reviews.shape, test_reviews.shape)
print("train items:", len(seen_train_business_ids))
print("cold-start items:", len(cold_start_business_ids))
print("zero-train items:", len(zero_train_business_ids))

if len(summary_df) > 0:
    display(summary_df)


Saved data loaded
DATA_DIR: /content/yelp_philly_processed
businesses: (3527, 14)
reviews: (511311, 11)
users: (178369, 12)
train/val/test: (357928, 13) (27914, 13) (17858, 13)
train items: 2890
cold-start items: 327
zero-train items: 637


,split,rows,unique_users,unique_businesses,date_min,date_max,cold_start_rows,cold_start_businesses,zero_train_rows
0,train,357928,124626,2890,2005-02-16,2018-08-24,1147,327,0
1,val,27914,9900,2404,2018-08-25,2019-11-22,634,193,0
2,test,17858,6647,2225,2019-11-23,2022-01-19,381,153,0


## Content-Based Metadata Baseline - Pranav

This baseline represents each restaurant using Yelp metadata such as categories, city, state, postal code, attributes, and hours. For each user, it builds a profile from restaurants they positively reviewed in training, then recommends unseen restaurants with similar metadata.

In [ ]:
# =========================================================
# Content-Based Metadata Baseline - Pranav
# =========================================================
import os
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

CONTENT_TOPN = 100
RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

if "businesses" not in globals():
    businesses = pd.read_csv(os.path.join(DATA_DIR, "businesses.csv"))

if "train_reviews" not in globals():
    SPLIT_DIR = os.path.join(DATA_DIR, "splits_temporal")
    train_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "train_reviews.csv"))
    val_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "val_reviews.csv"))
    test_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "test_reviews.csv"))

candidate_items = sorted(train_reviews["business_id"].unique())
candidate_set = set(candidate_items)

item_meta = businesses[businesses["business_id"].isin(candidate_set)].copy()
item_meta = item_meta.drop_duplicates("business_id").reset_index(drop=True)

def flatten_json_text(x):
    if pd.isna(x) or x in ["None", "nan", ""]:
        return ""
    try:
        obj = json.loads(x) if isinstance(x, str) else x
    except:
        return str(x)

    parts = []

    def walk(v):
        if isinstance(v, dict):
            for key, val in v.items():
                parts.append(str(key))
                walk(val)
        elif isinstance(v, list):
            for val in v:
                walk(val)
        else:
            parts.append(str(v))

    walk(obj)
    return " ".join(parts)

def build_metadata_text(row):
    fields = [
        row.get("categories", ""),
        row.get("city", ""),
        row.get("state", ""),
        row.get("postal_code", ""),
        flatten_json_text(row.get("attributes_json", "")),
        flatten_json_text(row.get("hours_json", ""))
    ]
    return " ".join([str(x) for x in fields if pd.notna(x)])

item_meta["metadata_text"] = item_meta.apply(build_metadata_text, axis=1)

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english"
)

item_matrix = vectorizer.fit_transform(item_meta["metadata_text"])
item_matrix = normalize(item_matrix)

content_items = item_meta["business_id"].tolist()
item_to_idx = {bid: i for i, bid in enumerate(content_items)}

train_pos = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS].copy()
train_pos = train_pos[train_pos["business_id"].isin(item_to_idx)].copy()

user_positive_items = (
    train_pos.groupby("user_id")["business_id"]
    .apply(list)
    .to_dict()
)

seen_map = make_seen_map(train_reviews)

popular_items = (
    train_reviews.groupby("business_id")
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)

def fill_with_popular(recs, seen_items, topn=CONTENT_TOPN):
    out = []
    used = set()

    for item in recs:
        if item in used or item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == topn:
            return out

    for item in popular_items:
        if item in used or item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == topn:
            break

    return out

def content_recs_for_users(user_list, topn=CONTENT_TOPN):
    rec_map = {}

    for uid in user_list:
        seen_items = seen_map.get(uid, set())
        liked_items = user_positive_items.get(uid, [])

        liked_idxs = [item_to_idx[x] for x in liked_items if x in item_to_idx]

        if len(liked_idxs) == 0:
            rec_map[uid] = fill_with_popular([], seen_items, topn)
            continue

        user_profile = item_matrix[liked_idxs].mean(axis=0)
        user_profile = normalize(np.asarray(user_profile))

        scores = np.asarray(user_profile @ item_matrix.T).ravel()

        for item in seen_items:
            idx = item_to_idx.get(item)
            if idx is not None:
                scores[idx] = -np.inf

        take_n = min(topn * 5, len(content_items))
        best_idx = np.argpartition(-scores, take_n - 1)[:take_n]
        best_idx = best_idx[np.argsort(-scores[best_idx])]

        raw_recs = [content_items[i] for i in best_idx if np.isfinite(scores[i])]
        rec_map[uid] = fill_with_popular(raw_recs, seen_items, topn)

    return rec_map

val_users = sorted(val_reviews["user_id"].unique())
test_users = sorted(test_reviews["user_id"].unique())

content_val_recs = content_recs_for_users(val_users, topn=CONTENT_TOPN)
content_test_recs = content_recs_for_users(test_users, topn=CONTENT_TOPN)

content_val_scores = evaluate_model(
    "content_metadata_baseline",
    content_val_recs,
    val_reviews,
    split_name="val",
    k_list=[5, 10, 20]
)

content_test_scores = evaluate_model(
    "content_metadata_baseline",
    content_test_recs,
    test_reviews,
    split_name="test",
    k_list=[5, 10, 20]
)

content_scores = pd.concat(
    [content_val_scores, content_test_scores],
    ignore_index=True
)

content_scores.to_csv(
    os.path.join(RESULTS_DIR, "content_metadata_scores.csv"),
    index=False
)

content_k10_table = content_scores[
    (content_scores["split"] == "val") &
    (content_scores["k"] == 10)
].copy()

content_k10_table.to_csv(
    os.path.join(RESULTS_DIR, "content_metadata_val_k10_table.csv"),
    index=False
)

print("saved:", os.path.join(RESULTS_DIR, "content_metadata_scores.csv"))
print("saved:", os.path.join(RESULTS_DIR, "content_metadata_val_k10_table.csv"))

display(content_scores)
display(content_k10_table)

## Matrix Factorization Baseline - Anunay

 It follows the latent factor idea from class: build a user-restaurant interaction matrix from positive training reviews, factor it into lower-dimensional user and restaurant vectors, then recommend unseen restaurants using user-vector dot item-vector scores.


In [ ]:
# =========================================================
# Matrix factorization baseline with TruncatedSVD
# =========================================================
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

MF_FACTORS = 64
MF_TOPN = 100
RANDOM_SEED = 172
RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Positive review = user liked the restaurant enough to use as implicit feedback.
train_pos = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS].copy()

# If a user reviewed the same restaurant more than once, one positive signal is enough.
train_pos = train_pos[["user_id", "business_id"]].drop_duplicates()

mf_users = sorted(train_reviews["user_id"].unique())
mf_items = sorted(train_reviews["business_id"].unique())

user_to_row = {uid: i for i, uid in enumerate(mf_users)}
item_to_col = {bid: i for i, bid in enumerate(mf_items)}
row_to_user = {i: uid for uid, i in user_to_row.items()}
col_to_item = {i: bid for bid, i in item_to_col.items()}

row_idx = train_pos["user_id"].map(user_to_row).to_numpy()
col_idx = train_pos["business_id"].map(item_to_col).to_numpy()
data = np.ones(len(train_pos), dtype=np.float32)

user_item_mat = csr_matrix(
    (data, (row_idx, col_idx)),
    shape=(len(mf_users), len(mf_items))
)

print("matrix shape:", user_item_mat.shape)
print("positive train pairs:", user_item_mat.nnz)

mf = TruncatedSVD(n_components=MF_FACTORS, random_state=RANDOM_SEED)
user_factors = mf.fit_transform(user_item_mat)
item_factors = mf.components_.T

print("latent factors:", MF_FACTORS)
print("variance captured:", round(float(mf.explained_variance_ratio_.sum()), 4))

# Popularity fallback keeps recommendations full for users with weak/empty positive history.
train_pop_items = (
    train_reviews.groupby("business_id")
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)

seen_map = make_seen_map(train_reviews)


def fill_with_popular(recs, seen_items, topn=MF_TOPN):
    out = []
    used = set()

    for item in recs:
        if item in used or item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == topn:
            return out

    for item in train_pop_items:
        if item in used or item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == topn:
            break

    return out


def mf_recs_for_users(user_list, topn=MF_TOPN):
    rec_map = {}

    for uid in user_list:
        seen_items = seen_map.get(uid, set())

        if uid not in user_to_row:
            rec_map[uid] = fill_with_popular([], seen_items, topn=topn)
            continue

        uidx = user_to_row[uid]
        scores = user_factors[uidx] @ item_factors.T

        for item in seen_items:
            col = item_to_col.get(item)
            if col is not None:
                scores[col] = -np.inf

        take_n = min(topn * 3, len(mf_items))
        best_cols = np.argpartition(-scores, take_n - 1)[:take_n]
        best_cols = best_cols[np.argsort(-scores[best_cols])]
        raw_recs = [col_to_item[c] for c in best_cols if np.isfinite(scores[c])]

        rec_map[uid] = fill_with_popular(raw_recs, seen_items, topn=topn)

    return rec_map


val_users = sorted(val_reviews["user_id"].unique())
test_users = sorted(test_reviews["user_id"].unique())

mf_val_recs = mf_recs_for_users(val_users, topn=MF_TOPN)
mf_test_recs = mf_recs_for_users(test_users, topn=MF_TOPN)

mf_val_scores = evaluate_model(
    "matrix_factorization_svd",
    mf_val_recs,
    val_reviews,
    split_name="val",
    k_list=[5, 10, 20]
)

mf_test_scores = evaluate_model(
    "matrix_factorization_svd",
    mf_test_recs,
    test_reviews,
    split_name="test",
    k_list=[5, 10, 20]
)

mf_scores = pd.concat([mf_val_scores, mf_test_scores], ignore_index=True)
mf_scores.to_csv(os.path.join(RESULTS_DIR, "matrix_factorization_scores.csv"), index=False)

print("saved:", os.path.join(RESULTS_DIR, "matrix_factorization_scores.csv"))
display(mf_scores)


# Popularity - Aaron

In [ ]:
# =========================================================
# Popularity Baseline
# =========================================================
# Score every business with a Bayesian-smoothed average rating then provide the ranks

# Bayesian avg = (C * m + n * r) / (C + n)
#   m = global mean rating across all train reviews
#   n = number of train reviews for this business
#   r = mean train rating for this business
#   C = smoothing constant — higher means more shrinkage toward global mean
#       prevents biz w/ just one high star review from dominating
# =========================================================

POP_TOPN     = 100   # recommendation list length per user, matches MF_TOPN = 100
POP_SMOOTH_C = 10    # Bayesian smoothing constant

# Compute per-business stats from training reviews, defined in equation above
train_biz_stats = (
    train_reviews
    .groupby("business_id")["stars"]
    .agg(train_count="count", train_mean_stars="mean")
    .reset_index()
)

# get global mean from all stars
global_mean = train_reviews["stars"].mean()

# implementing equation
train_biz_stats["bayes_score"] = (
    (POP_SMOOTH_C * global_mean + train_biz_stats["train_count"] * train_biz_stats["train_mean_stars"])
    / (POP_SMOOTH_C + train_biz_stats["train_count"])
)

# apply to cold start businesses
train_biz_stats["is_cold_start"] = train_biz_stats["business_id"].isin(cold_start_business_ids)

# Build sorted recommendation pools
# Cold-start items first (these are what we want to surface to users)
cold_start_ranked = (
    train_biz_stats[train_biz_stats["is_cold_start"]]
    .sort_values("bayes_score", ascending=False)["business_id"]
    .tolist()
)

# Warm items as fallback in case the cold-start pool runs dry
warm_ranked = (
    train_biz_stats[~train_biz_stats["is_cold_start"]]
    .sort_values("bayes_score", ascending=False)["business_id"]
    .tolist()
)

seen_map = make_seen_map(train_reviews)


COLD_START_RATIO = 0.3   # fraction of each recommendation list that is cold-start

def pop_recs_for_user(uid, topn=POP_TOPN):
    """Recommend top-N businesses, prioritizing cold-start ones the user hasn't seen."""
    seen = seen_map.get(uid, set())
    recs, used = [], set()

    for bid in cold_start_ranked:
        if bid not in seen and bid not in used:
            recs.append(bid)
            used.add(bid)
        if len(recs) == topn:
            return recs

    for bid in warm_ranked:
        if bid not in seen and bid not in used:
            recs.append(bid)
            used.add(bid)
        if len(recs) == topn:
            break

    return recs


# Generate recommendations
val_users  = sorted(val_reviews["user_id"].unique())
test_users = sorted(test_reviews["user_id"].unique())

pop_val_recs  = {uid: pop_recs_for_user(uid) for uid in val_users}
pop_test_recs = {uid: pop_recs_for_user(uid) for uid in test_users}

# Evaluate
pop_val_scores  = evaluate_model("popularity_cold_start", pop_val_recs,  val_reviews,  split_name="val",  k_list=[5, 10, 20])
pop_test_scores = evaluate_model("popularity_cold_start", pop_test_recs, test_reviews, split_name="test", k_list=[5, 10, 20])

pop_scores = pd.concat([pop_val_scores, pop_test_scores], ignore_index=True)
pop_scores.to_csv(os.path.join(RESULTS_DIR, "popularity_cold_start_scores.csv"), index=False)

print(f"Global mean rating (train): {global_mean:.3f}")
print(f"Cold-start pool size: {len(cold_start_ranked)} businesses")
print(f"Warm fallback pool size: {len(warm_ranked)} businesses")
print()
display(pop_scores)

## FIDR-Inspired Hybrid Re-Ranker

This is our project method. The FIDR paper argues that interaction-only retrieval misses cold-start and long-tail items, so it adds an item-content structure next to the user-interaction structure. We use the same idea in a simpler Colab-friendly form: matrix factorization gives the interaction signal, TF-IDF metadata gives the item-content signal, and a small cold-start term helps low-review restaurants enter the ranking.


In [ ]:
# =========================================================
# FIDR-inspired hybrid re-ranker
# =========================================================
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

HYBRID_TOPN = 100
HYBRID_FACTORS = 64
HYBRID_RANDOM_SEED = 172

DATA_DIR = "/content/yelp_philly_processed" # Added this line
RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. Make sure the interaction side exists.
#    This is the User-Structure-style signal from FIDR.
# ---------------------------------------------------------
needed_mf_vars = ["user_factors", "item_factors", "user_to_row", "item_to_col"]
if any(name not in globals() for name in needed_mf_vars):
    print("MF variables not found, training the SVD interaction side here.")

    train_pos = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS].copy()
    train_pos = train_pos[["user_id", "business_id"]].drop_duplicates()

    mf_users = sorted(train_reviews["user_id"].unique())
    mf_items = sorted(train_reviews["business_id"].unique())

    user_to_row = {uid: i for i, uid in enumerate(mf_users)}
    item_to_col = {bid: i for i, bid in enumerate(mf_items)}
    row_to_user = {i: uid for uid, i in user_to_row.items()}
    col_to_item = {i: bid for bid, i in item_to_col.items()}

    row_idx = train_pos["user_id"].map(user_to_row).to_numpy()
    col_idx = train_pos["business_id"].map(item_to_col).to_numpy()
    data = np.ones(len(train_pos), dtype=np.float32)

    user_item_mat = csr_matrix(
        (data, (row_idx, col_idx)),
        shape=(len(mf_users), len(mf_items))
    )

    mf = TruncatedSVD(n_components=HYBRID_FACTORS, random_state=HYBRID_RANDOM_SEED)
    user_factors = mf.fit_transform(user_item_mat)
    item_factors = mf.components_.T

# Candidate restaurants are the restaurants seen in train.
# This matches our evaluation setup, where zero-train restaurants are removed from val/test.
candidate_items = sorted(train_reviews["business_id"].unique())
candidate_pos = {bid: i for i, bid in enumerate(candidate_items)}
candidate_cols = np.array([item_to_col[bid] for bid in candidate_items])

seen_map = make_seen_map(train_reviews)

# ---------------------------------------------------------
# 2. Build the item-content side from Yelp metadata.
#    This is the Item-Structure-style signal from FIDR.
# ---------------------------------------------------------
item_meta = businesses[[
    "business_id", "name", "categories", "attributes_json", "postal_code"
]].copy()

item_meta = item_meta[item_meta["business_id"].isin(candidate_pos)].copy()
item_meta = item_meta.set_index("business_id").loc[candidate_items].reset_index()


def clean_text_piece(x):
    if pd.isna(x):
        return ""
    return str(x).replace("_", " ").replace(",", " ").replace("{", " ").replace("}", " ")


item_meta["meta_text"] = (
    item_meta["name"].map(clean_text_piece) + " " +
    item_meta["categories"].map(clean_text_piece) + " " +
    item_meta["attributes_json"].map(clean_text_piece) + " " +
    item_meta["postal_code"].map(clean_text_piece)
)

content_vec = TfidfVectorizer(
    max_features=12000,
    min_df=1,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
    norm="l2"
)

item_content_mat = content_vec.fit_transform(item_meta["meta_text"])
print("content matrix:", item_content_mat.shape)

# Liked history for making user content profiles.
train_likes = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS]
liked_map = (
    train_likes.groupby("user_id")["business_id"]
    .apply(list)
    .to_dict()
)

# ---------------------------------------------------------
# 3. Train-only item quality and cold-start terms.
# ---------------------------------------------------------
train_biz_stats = (
    train_reviews.groupby("business_id")["stars"]
    .agg(train_count="count", train_mean_stars="mean")
    .reset_index()
)

global_mean = train_reviews["stars"].mean()
SMOOTH_C = 10
train_biz_stats["bayes_score"] = (
    (SMOOTH_C * global_mean + train_biz_stats["train_count"] * train_biz_stats["train_mean_stars"])
    / (SMOOTH_C + train_biz_stats["train_count"])
)

item_stats = pd.DataFrame({"business_id": candidate_items})
item_stats = item_stats.merge(train_biz_stats, on="business_id", how="left")
item_stats["train_count"] = item_stats["train_count"].fillna(0)
item_stats["train_mean_stars"] = item_stats["train_mean_stars"].fillna(global_mean)
item_stats["bayes_score"] = item_stats["bayes_score"].fillna(global_mean)
item_stats["is_cold_start"] = item_stats["business_id"].isin(cold_start_business_ids).astype(float)


def minmax(x):
    x = np.asarray(x, dtype=np.float32)
    lo = np.nanmin(x)
    hi = np.nanmax(x)
    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)
    return (x - lo) / (hi - lo)


bayes_n = minmax(item_stats["bayes_score"].to_numpy())
pop_n = minmax(np.log1p(item_stats["train_count"].to_numpy()))
low_count_n = 1.0 - pop_n
cold_n = item_stats["is_cold_start"].to_numpy(dtype=np.float32)

# These weights are deliberately transparent for the paper.
# They mirror FIDR's idea: combine interaction structure + item-content structure + cold-start attention.
hybrid_weights = {
    "mf_score": 0.42,
    "content_score": 0.22,
    "bayes_score": 0.15,
    "popularity": 0.10,
    "cold_start": 0.08,
    "low_count": 0.03,
}

weight_table = pd.DataFrame(
    [{"signal": k, "weight": v} for k, v in hybrid_weights.items()]
)


def norm_user_scores(scores):
    scores = np.asarray(scores, dtype=np.float32)
    finite = np.isfinite(scores)
    out = np.zeros_like(scores, dtype=np.float32)
    if finite.sum() == 0:
        return out
    vals = scores[finite]
    lo = vals.min()
    hi = vals.max()
    if hi <= lo:
        return out
    out[finite] = (vals - lo) / (hi - lo)
    return out


def content_scores_for_user(uid):
    liked_items = liked_map.get(uid, [])
    item_rows = [candidate_pos[item] for item in liked_items if item in candidate_pos]

    if not item_rows:
        return np.zeros(len(candidate_items), dtype=np.float32)

    user_profile = item_content_mat[item_rows].mean(axis=0)
    scores = user_profile @ item_content_mat.T
    return np.asarray(scores).ravel().astype(np.float32)


def mf_scores_for_user(uid):
    if uid not in user_to_row:
        return np.zeros(len(candidate_items), dtype=np.float32)

    raw_scores = user_factors[user_to_row[uid]] @ item_factors.T
    return raw_scores[candidate_cols].astype(np.float32)


def hybrid_recs_for_user(uid, topn=HYBRID_TOPN):
    seen = seen_map.get(uid, set())

    mf_n = norm_user_scores(mf_scores_for_user(uid))
    content_n = norm_user_scores(content_scores_for_user(uid))

    final_score = (
        hybrid_weights["mf_score"] * mf_n +
        hybrid_weights["content_score"] * content_n +
        hybrid_weights["bayes_score"] * bayes_n +
        hybrid_weights["popularity"] * pop_n +
        hybrid_weights["cold_start"] * cold_n +
        hybrid_weights["low_count"] * low_count_n
    )

    for item in seen:
        pos = candidate_pos.get(item)
        if pos is not None:
            final_score[pos] = -np.inf

    take_n = min(topn * 4, len(candidate_items))
    best_pos = np.argpartition(-final_score, take_n - 1)[:take_n]
    best_pos = best_pos[np.argsort(-final_score[best_pos])]

    recs = []
    used = set()
    for pos in best_pos:
        item = candidate_items[pos]
        if item in used or item in seen:
            continue
        recs.append(item)
        used.add(item)
        if len(recs) == topn:
            break

    return recs


def hybrid_recs_for_users(user_list, topn=HYBRID_TOPN):
    rec_map = {}
    for n, uid in enumerate(user_list, start=1):
        rec_map[uid] = hybrid_recs_for_user(uid, topn=topn)
        if n % 1000 == 0:
            print("hybrid users finished:", n)
    return rec_map


val_users = sorted(val_reviews["user_id"].unique())
test_users = sorted(test_reviews["user_id"].unique())

print("making validation hybrid recs...")
hybrid_val_recs = hybrid_recs_for_users(val_users, topn=HYBRID_TOPN)

print("making test hybrid recs...")
hybrid_test_recs = hybrid_recs_for_users(test_users, topn=HYBRID_TOPN)

hybrid_val_scores = evaluate_model(
    "fidr_inspired_hybrid",
    hybrid_val_recs,
    val_reviews,
    split_name="val",
    k_list=[5, 10, 20]
)

hybrid_test_scores = evaluate_model(
    "fidr_inspired_hybrid",
    hybrid_test_recs,
    test_reviews,
    split_name="test",
    k_list=[5, 10, 20]
)

hybrid_scores = pd.concat([hybrid_val_scores, hybrid_test_scores], ignore_index=True)
hybrid_scores.to_csv(os.path.join(RESULTS_DIR, "fidr_inspired_hybrid_scores.csv"), index=False)

print("saved:", os.path.join(RESULTS_DIR, "fidr_inspired_hybrid_scores.csv"))
print("Hybrid weights:")
display(weight_table)
display(hybrid_scores)


content matrix: (2890, 12000)
making validation hybrid recs...
hybrid users finished: 1000
hybrid users finished: 2000
hybrid users finished: 3000
hybrid users finished: 4000
hybrid users finished: 5000
hybrid users finished: 6000
hybrid users finished: 7000
hybrid users finished: 8000
hybrid users finished: 9000
making test hybrid recs...
hybrid users finished: 1000
hybrid users finished: 2000
hybrid users finished: 3000
hybrid users finished: 4000
hybrid users finished: 5000
hybrid users finished: 6000
saved: /content/yelp_philly_processed/model_results/fidr_inspired_hybrid_scores.csv
Hybrid weights:


,signal,weight
0,mf_score,0.42
1,content_score,0.22
2,bayes_score,0.15
3,popularity,0.10
4,cold_start,0.08
5,low_count,0.03


,model,split,group,k,users_eval,hit@k,recall@k,mrr@k,ndcg@k
0,fidr_inspired_hybrid,val,all_positive,5,7551,0.046087,0.017243,0.022489,0.015201
1,fidr_inspired_hybrid,val,cold_start_positive,5,363,0.005510,0.004132,0.003444,0.003482
2,fidr_inspired_hybrid,val,all_positive,10,7551,0.072441,0.029362,0.025944,0.019035
3,fidr_inspired_hybrid,val,cold_start_positive,10,363,0.011019,0.009642,0.004247,0.005333
4,fidr_inspired_hybrid,val,all_positive,20,7551,0.114157,0.049907,0.028789,0.025008
5,fidr_inspired_hybrid,val,cold_start_positive,20,363,0.030303,0.028926,0.005483,0.010085
6,fidr_inspired_hybrid,test,all_positive,5,5003,0.037178,0.017011,0.019568,0.014476
7,fidr_inspired_hybrid,test,cold_start_positive,5,183,0.005464,0.005464,0.001366,0.002353
8,fidr_inspired_hybrid,test,all_positive,10,5003,0.062562,0.028785,0.022917,0.018382
9,fidr_inspired_hybrid,test,cold_start_positive,10,183,0.010929,0.010929,0.002277,0.004300


In [ ]:
# =========================================================
# TANMMAY — Cell 1 of 2
# Pure Content-Based Baseline (TF-IDF)
# =========================================================
# Recommend restaurants using ONLY item metadata —
# no interaction signal at all. User profile = average
# TF-IDF vector of restaurants they liked in train.
# Scoring new items = cosine similarity to that profile.
#
# Why this matters for the paper:
#   Isolates the content signal. When we compare to FIDR
#   hybrid, any gain proves the interaction signal is
#   adding real value beyond content alone.
# =========================================================

from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

CONTENT_TOPN = 100

RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)


# ---------------------------------------------------------
# Build item metadata text for ALL candidate items
# (same logic as FIDR hybrid so signals are comparable)
# ---------------------------------------------------------
def clean_text_piece(x):
    if pd.isna(x):
        return ""
    return str(x).replace("_", " ").replace(",", " ").replace("{", " ").replace("}", " ")


candidate_items_cb = sorted(train_reviews["business_id"].unique())
candidate_pos_cb   = {bid: i for i, bid in enumerate(candidate_items_cb)}

item_meta_cb = businesses[
    ["business_id", "name", "categories", "attributes_json", "postal_code"]
].copy()

item_meta_cb = (
    item_meta_cb[item_meta_cb["business_id"].isin(candidate_pos_cb)]
    .set_index("business_id")
    .loc[candidate_items_cb]
    .reset_index()
)

item_meta_cb["meta_text"] = (
    item_meta_cb["name"].map(clean_text_piece)       + " " +
    item_meta_cb["categories"].map(clean_text_piece)  + " " +
    item_meta_cb["attributes_json"].map(clean_text_piece) + " " +
    item_meta_cb["postal_code"].map(clean_text_piece)
)

cb_vectorizer = TfidfVectorizer(
    max_features=12000,
    min_df=1,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
    norm="l2"          # L2-norm means dot product == cosine similarity
)

cb_item_mat = cb_vectorizer.fit_transform(item_meta_cb["meta_text"])
print("content matrix:", cb_item_mat.shape)


# ---------------------------------------------------------
# User profile = mean of liked items' TF-IDF vectors
# Fall back to global mean vector for users with no likes
# ---------------------------------------------------------
train_likes_cb = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS]
liked_map_cb   = (
    train_likes_cb.groupby("user_id")["business_id"]
    .apply(list)
    .to_dict()
)

global_profile_cb = cb_item_mat.mean(axis=0)  # (1, n_features)
seen_map_cb       = make_seen_map(train_reviews)


def cb_recs_for_user(uid, topn=CONTENT_TOPN):
    seen  = seen_map_cb.get(uid, set())
    liked = [b for b in liked_map_cb.get(uid, []) if b in candidate_pos_cb]

    if liked:
        rows    = [candidate_pos_cb[b] for b in liked]
        profile = cb_item_mat[rows].mean(axis=0)          # (1, n_features)
    else:
        profile = global_profile_cb

    scores = np.asarray(profile @ cb_item_mat.T).ravel()  # cosine sim to every item

    # Suppress already-seen restaurants
    for item in seen:
        pos = candidate_pos_cb.get(item)
        if pos is not None:
            scores[pos] = -np.inf

    take_n   = min(topn * 3, len(candidate_items_cb))
    best_pos = np.argpartition(-scores, take_n - 1)[:take_n]
    best_pos = best_pos[np.argsort(-scores[best_pos])]

    recs, used = [], set()
    for pos in best_pos:
        item = candidate_items_cb[pos]
        if item in used or item in seen:
            continue
        recs.append(item)
        used.add(item)
        if len(recs) == topn:
            break
    return recs


val_users_cb  = sorted(val_reviews["user_id"].unique())
test_users_cb = sorted(test_reviews["user_id"].unique())

print("making content-based val recs...")
cb_val_recs  = {uid: cb_recs_for_user(uid) for uid in val_users_cb}

print("making content-based test recs...")
cb_test_recs = {uid: cb_recs_for_user(uid) for uid in test_users_cb}

cb_val_scores  = evaluate_model(
    "content_based_tfidf", cb_val_recs,  val_reviews,  split_name="val",  k_list=[5, 10, 20]
)
cb_test_scores = evaluate_model(
    "content_based_tfidf", cb_test_recs, test_reviews, split_name="test", k_list=[5, 10, 20]
)

cb_scores = pd.concat([cb_val_scores, cb_test_scores], ignore_index=True)
cb_scores.to_csv(os.path.join(RESULTS_DIR, "content_based_tfidf_scores.csv"), index=False)

print("saved:", os.path.join(RESULTS_DIR, "content_based_tfidf_scores.csv"))
display(cb_scores)


In [ ]:
# =========================================================
# TANMMAY — Cell 2 of 2
# LightFM (BPR + Item Features)
# =========================================================
# LightFM is a proper ranking-optimised CF model. Unlike
# SVD (which minimises reconstruction error), LightFM
# trains a Bayesian Personalised Ranking loss — it directly
# optimises the ranking of positive items over unobserved
# ones.
#
# The key cold-start advantage: LightFM takes item feature
# vectors as input alongside interactions. Cold-start
# restaurants (1–5 train reviews) get their embeddings
# anchored to their category/metadata features, so they
# are not invisible to the model even with sparse signal.
#
# Paper connection: this implements the core idea from
# CCFCRec (WWW 2023) — bridging interaction and content
# signals through a unified embedding space.
# =========================================================

!pip -q install lightfm

import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset
from scipy.sparse import csr_matrix

LFM_COMPONENTS  = 64
LFM_EPOCHS      = 30
LFM_TOPN        = 100
LFM_RANDOM_SEED = 172

RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)


# ---------------------------------------------------------
# 1. Build category feature list for items
#    Each restaurant gets a bag-of-categories feature vec.
#    Cold-start items get real features even with almost
#    no interactions.
# ---------------------------------------------------------
def parse_categories(cat_str):
    if pd.isna(cat_str) or not cat_str:
        return []
    return [c.strip().lower() for c in str(cat_str).split(",")]


all_businesses_lfm = businesses[
    businesses["business_id"].isin(set(train_reviews["business_id"]))
].copy()

all_businesses_lfm["cat_list"] = all_businesses_lfm["categories"].apply(parse_categories)

# Collect every unique category tag
all_cats = sorted({cat for cats in all_businesses_lfm["cat_list"] for cat in cats})
print(f"unique category tags: {len(all_cats)}")


# ---------------------------------------------------------
# 2. Fit LightFM Dataset — maps string IDs to int indices
# ---------------------------------------------------------
lfm_dataset = Dataset()

lfm_dataset.fit(
    users=train_reviews["user_id"].unique(),
    items=all_businesses_lfm["business_id"].unique(),
    item_features=all_cats
)

n_users, n_items = lfm_dataset.interactions_shape()
print(f"LightFM dataset: {n_users} users, {n_items} items")


# ---------------------------------------------------------
# 3. Build interaction matrix (positive train reviews only)
# ---------------------------------------------------------
train_pos_lfm = (
    train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS]
    [["user_id", "business_id"]]
    .drop_duplicates()
)

(lfm_interactions, _) = lfm_dataset.build_interactions(
    [(row.user_id, row.business_id) for row in train_pos_lfm.itertuples(index=False)]
)

print("interactions matrix:", lfm_interactions.shape,
      "| positives:", lfm_interactions.nnz)


# ---------------------------------------------------------
# 4. Build item feature matrix from categories
# ---------------------------------------------------------
lfm_item_features = lfm_dataset.build_item_features(
    (
        (row.business_id, row.cat_list)
        for row in all_businesses_lfm[["business_id", "cat_list"]].itertuples(index=False)
    )
)

print("item features matrix:", lfm_item_features.shape)


# ---------------------------------------------------------
# 5. Train LightFM with BPR loss
# ---------------------------------------------------------
lfm_model = LightFM(
    no_components=LFM_COMPONENTS,
    loss="bpr",
    random_state=LFM_RANDOM_SEED
)

print(f"training LightFM for {LFM_EPOCHS} epochs...")
lfm_model.fit(
    lfm_interactions,
    item_features=lfm_item_features,
    epochs=LFM_EPOCHS,
    num_threads=2,
    verbose=False
)
print("training complete.")


# ---------------------------------------------------------
# 6. Build lookup tables for prediction
# ---------------------------------------------------------
user_id_map, _, item_id_map, _ = lfm_dataset.mapping()
# user_id_map: str user_id -> int index
# item_id_map: str business_id -> int index

int_to_item = {v: k for k, v in item_id_map.items()}

# Only score items that exist in our candidate pool
# (items seen in train — mirrors the evaluation setup)
candidate_items_lfm = sorted(train_reviews["business_id"].unique())
candidate_int_ids   = np.array([
    item_id_map[bid] for bid in candidate_items_lfm if bid in item_id_map
], dtype=np.int32)
candidate_bids      = [int_to_item[i] for i in candidate_int_ids]

seen_map_lfm = make_seen_map(train_reviews)


# ---------------------------------------------------------
# 7. Generate top-N recommendations per user
# ---------------------------------------------------------
def lfm_recs_for_user(uid, topn=LFM_TOPN):
    seen = seen_map_lfm.get(uid, set())

    if uid not in user_id_map:
        # Unknown user at eval time — fall back to popularity
        return [b for b in train_pop_items if b not in seen][:topn]

    uidx = user_id_map[uid]

    scores = lfm_model.predict(
        user_ids=np.full(len(candidate_int_ids), uidx, dtype=np.int32),
        item_ids=candidate_int_ids,
        item_features=lfm_item_features
    )

    # Suppress seen items
    for i, bid in enumerate(candidate_bids):
        if bid in seen:
            scores[i] = -np.inf

    take_n   = min(topn * 3, len(candidate_bids))
    best_pos = np.argpartition(-scores, take_n - 1)[:take_n]
    best_pos = best_pos[np.argsort(-scores[best_pos])]

    recs, used = [], set()
    for pos in best_pos:
        item = candidate_bids[pos]
        if item in used or item in seen:
            continue
        recs.append(item)
        used.add(item)
        if len(recs) == topn:
            break
    return recs


def lfm_recs_for_users(user_list, topn=LFM_TOPN):
    rec_map = {}
    for n, uid in enumerate(user_list, start=1):
        rec_map[uid] = lfm_recs_for_user(uid, topn=topn)
        if n % 1000 == 0:
            print(f"LightFM users done: {n}")
    return rec_map


# Popularity fallback (needed for unknown users above)
if "train_pop_items" not in globals():
    train_pop_items = (
        train_reviews.groupby("business_id")
        .size()
        .sort_values(ascending=False)
        .index.tolist()
    )

val_users_lfm  = sorted(val_reviews["user_id"].unique())
test_users_lfm = sorted(test_reviews["user_id"].unique())

print("making LightFM val recs...")
lfm_val_recs  = lfm_recs_for_users(val_users_lfm,  topn=LFM_TOPN)

print("making LightFM test recs...")
lfm_test_recs = lfm_recs_for_users(test_users_lfm, topn=LFM_TOPN)


# ---------------------------------------------------------
# 8. Evaluate and save
# ---------------------------------------------------------
lfm_val_scores  = evaluate_model(
    "lightfm_bpr", lfm_val_recs,  val_reviews,  split_name="val",  k_list=[5, 10, 20]
)
lfm_test_scores = evaluate_model(
    "lightfm_bpr", lfm_test_recs, test_reviews, split_name="test", k_list=[5, 10, 20]
)

lfm_scores = pd.concat([lfm_val_scores, lfm_test_scores], ignore_index=True)
lfm_scores.to_csv(os.path.join(RESULTS_DIR, "lightfm_bpr_scores.csv"), index=False)

print("saved:", os.path.join(RESULTS_DIR, "lightfm_bpr_scores.csv"))
display(lfm_scores)

## B2P-Inspired Personalized Popularity

This method adapts the RecSys 2023 B2P paper. The paper's main idea is that popularity is strong for sparse/cold users, but raw popularity should be personalized using item metadata and bootstrapped with a collaborative model for warmer users. Here we use user-history buckets, metadata similarity, train-only Bayesian popularity, and an MF warm-user component.


In [ ]:
# =========================================================
# B2P-inspired personalized popularity
# =========================================================
# Paper idea: popularity is useful for cold/sparse users, but it should
# be personalized with item metadata and bootstrapped with CF for warm users.
# Our adaptation:
#   - group users by train history size
#   - compute popularity within each user-history stratum
#   - personalize it with TF-IDF restaurant metadata similarity
#   - switch/mix with matrix-factorization scores for warmer users
# =========================================================
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

B2P_TOPN = 100
B2P_FACTORS = 64
B2P_RANDOM_SEED = 172
B2P_COLD_USER_THETA = 5
RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

needed_b2p_helpers = ["make_seen_map", "evaluate_model"]
missing_helpers = [name for name in needed_b2p_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError("Run the shared evaluator cell before B2P. Missing: " + str(missing_helpers))

if "cold_start_business_ids" not in globals():
    cold_start_business_ids = set(cold_start_items["business_id"])

# ---------------------------------------------------------
# 1. Candidate items and basic maps
# ---------------------------------------------------------
candidate_items_b2p = sorted(train_reviews["business_id"].unique())
candidate_pos_b2p = {bid: i for i, bid in enumerate(candidate_items_b2p)}
seen_map_b2p = make_seen_map(train_reviews)

train_user_hist = train_reviews.groupby("user_id").size().to_dict()


def user_history_bucket(n):
    if n <= 1:
        return "n_1"
    if n <= 3:
        return "n_2_3"
    if n <= 5:
        return "n_4_5"
    if n <= 10:
        return "n_6_10"
    if n <= 20:
        return "n_11_20"
    return "n_21_plus"


# ---------------------------------------------------------
# 2. Stratum popularity: popularity among similar-history users
# ---------------------------------------------------------
train_pos_b2p = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS].copy()
train_pos_b2p = train_pos_b2p[["user_id", "business_id"]].drop_duplicates()
train_pos_b2p["hist_n"] = train_pos_b2p["user_id"].map(train_user_hist).fillna(0).astype(int)
train_pos_b2p["hist_bucket"] = train_pos_b2p["hist_n"].map(user_history_bucket)


def minmax_vec(x):
    x = np.asarray(x, dtype=np.float32)
    lo = np.nanmin(x)
    hi = np.nanmax(x)
    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)
    return (x - lo) / (hi - lo)


def count_vector(df):
    counts = np.zeros(len(candidate_items_b2p), dtype=np.float32)
    item_counts = df.groupby("business_id").size()
    for bid, c in item_counts.items():
        pos = candidate_pos_b2p.get(bid)
        if pos is not None:
            counts[pos] = c
    return minmax_vec(np.log1p(counts))


global_stratum_pop = count_vector(train_pos_b2p)
stratum_pop = {}
for bucket, part in train_pos_b2p.groupby("hist_bucket"):
    vec = count_vector(part)
    # If a bucket is too sparse, keep it from becoming all zeros.
    if vec.sum() == 0:
        vec = global_stratum_pop.copy()
    stratum_pop[bucket] = vec

print("B2P user-history buckets:", sorted(stratum_pop.keys()))

# ---------------------------------------------------------
# 3. Metadata representation for personalization
# ---------------------------------------------------------
def clean_b2p_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("_", " ").replace(",", " ").replace("{", " ").replace("}", " ")


item_meta_b2p = businesses[[
    "business_id", "name", "categories", "attributes_json", "postal_code"
]].copy()
item_meta_b2p = item_meta_b2p[item_meta_b2p["business_id"].isin(candidate_pos_b2p)].copy()
item_meta_b2p = item_meta_b2p.set_index("business_id").loc[candidate_items_b2p].reset_index()

item_meta_b2p["meta_text"] = (
    item_meta_b2p["name"].map(clean_b2p_text) + " " +
    item_meta_b2p["categories"].map(clean_b2p_text) + " " +
    item_meta_b2p["attributes_json"].map(clean_b2p_text) + " " +
    item_meta_b2p["postal_code"].map(clean_b2p_text)
)

b2p_vec = TfidfVectorizer(
    max_features=12000,
    min_df=1,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
    norm="l2"
)

b2p_item_mat = b2p_vec.fit_transform(item_meta_b2p["meta_text"])
print("B2P content matrix:", b2p_item_mat.shape)

liked_map_b2p = (
    train_pos_b2p.groupby("user_id")["business_id"]
    .apply(list)
    .to_dict()
)

# ---------------------------------------------------------
# 4. Train-only item quality and cold/long-tail signals
# ---------------------------------------------------------
train_biz_stats_b2p = (
    train_reviews.groupby("business_id")["stars"]
    .agg(train_count="count", train_mean_stars="mean")
    .reset_index()
)

global_mean_b2p = train_reviews["stars"].mean()
B2P_SMOOTH_C = 10
train_biz_stats_b2p["bayes_score"] = (
    (B2P_SMOOTH_C * global_mean_b2p + train_biz_stats_b2p["train_count"] * train_biz_stats_b2p["train_mean_stars"])
    / (B2P_SMOOTH_C + train_biz_stats_b2p["train_count"])
)

item_stats_b2p = pd.DataFrame({"business_id": candidate_items_b2p})
item_stats_b2p = item_stats_b2p.merge(train_biz_stats_b2p, on="business_id", how="left")
item_stats_b2p["train_count"] = item_stats_b2p["train_count"].fillna(0)
item_stats_b2p["bayes_score"] = item_stats_b2p["bayes_score"].fillna(global_mean_b2p)
item_stats_b2p["is_cold_start"] = item_stats_b2p["business_id"].isin(cold_start_business_ids).astype(float)

b2p_bayes_n = minmax_vec(item_stats_b2p["bayes_score"].to_numpy())
b2p_pop_n = minmax_vec(np.log1p(item_stats_b2p["train_count"].to_numpy()))
b2p_low_count_n = 1.0 - b2p_pop_n
b2p_cold_n = item_stats_b2p["is_cold_start"].to_numpy(dtype=np.float32)

# ---------------------------------------------------------
# 5. Collaborative warm-user component
# ---------------------------------------------------------
need_new_mf = any(name not in globals() for name in ["user_factors", "item_factors", "user_to_row", "item_to_col"])
if need_new_mf:
    print("MF variables not found, training B2P SVD warm-user component.")

    b2p_mf_users = sorted(train_reviews["user_id"].unique())
    b2p_mf_items = candidate_items_b2p
    b2p_user_to_row = {uid: i for i, uid in enumerate(b2p_mf_users)}
    b2p_item_to_col = {bid: i for i, bid in enumerate(b2p_mf_items)}

    row_idx = train_pos_b2p["user_id"].map(b2p_user_to_row).to_numpy()
    col_idx = train_pos_b2p["business_id"].map(b2p_item_to_col).to_numpy()
    data = np.ones(len(train_pos_b2p), dtype=np.float32)

    b2p_user_item_mat = csr_matrix(
        (data, (row_idx, col_idx)),
        shape=(len(b2p_mf_users), len(b2p_mf_items))
    )

    b2p_mf = TruncatedSVD(n_components=B2P_FACTORS, random_state=B2P_RANDOM_SEED)
    b2p_user_factors = b2p_mf.fit_transform(b2p_user_item_mat)
    b2p_item_factors = b2p_mf.components_.T
    b2p_candidate_cols = np.arange(len(candidate_items_b2p))
else:
    b2p_user_to_row = user_to_row
    b2p_item_to_col = item_to_col
    b2p_user_factors = user_factors
    b2p_item_factors = item_factors
    b2p_candidate_cols = np.array([b2p_item_to_col[bid] for bid in candidate_items_b2p])


def norm_user_scores(scores):
    scores = np.asarray(scores, dtype=np.float32)
    finite = np.isfinite(scores)
    out = np.zeros_like(scores, dtype=np.float32)
    if finite.sum() == 0:
        return out
    vals = scores[finite]
    lo = vals.min()
    hi = vals.max()
    if hi <= lo:
        return out
    out[finite] = (vals - lo) / (hi - lo)
    return out


def b2p_content_scores(uid):
    liked_items = liked_map_b2p.get(uid, [])
    rows = [candidate_pos_b2p[item] for item in liked_items if item in candidate_pos_b2p]

    if not rows:
        return np.zeros(len(candidate_items_b2p), dtype=np.float32)

    user_profile = b2p_item_mat[rows].mean(axis=0)
    scores = user_profile @ b2p_item_mat.T
    return np.asarray(scores).ravel().astype(np.float32)


def b2p_mf_scores(uid):
    if uid not in b2p_user_to_row:
        return np.zeros(len(candidate_items_b2p), dtype=np.float32)
    raw = b2p_user_factors[b2p_user_to_row[uid]] @ b2p_item_factors.T
    return raw[b2p_candidate_cols].astype(np.float32)


# Cold users lean more on B2P personalized popularity.
# Warm users are bootstrapped with more collaborative signal.
b2p_cold_user_weights = {
    "stratum_pop": 0.35,
    "content_match": 0.30,
    "bayes_score": 0.15,
    "global_pop": 0.05,
    "cold_start": 0.10,
    "low_count": 0.05,
    "mf_score": 0.00,
}

b2p_warm_user_weights = {
    "stratum_pop": 0.15,
    "content_match": 0.15,
    "bayes_score": 0.10,
    "global_pop": 0.05,
    "cold_start": 0.08,
    "low_count": 0.04,
    "mf_score": 0.43,
}

# b2p_warm_user_weights = {
#     "stratum_pop": 0.15,
#     "content_match": 0.15,
#     "bayes_score": 0.10,
#     "global_pop": 0.05,
#     "cold_start": 0.05,
#     "low_count": 0.02,
#     "mf_score": 0.48,
# }

b2p_weight_table = pd.DataFrame([
    {"user_group": "cold_or_sparse_user", "signal": k, "weight": v}
    for k, v in b2p_cold_user_weights.items()
] + [
    {"user_group": "warm_user", "signal": k, "weight": v}
    for k, v in b2p_warm_user_weights.items()
])


def b2p_recs_for_user(uid, topn=B2P_TOPN):
    seen = seen_map_b2p.get(uid, set())
    hist_n = int(train_user_hist.get(uid, 0))
    bucket = user_history_bucket(hist_n)

    weights = b2p_cold_user_weights if hist_n <= B2P_COLD_USER_THETA else b2p_warm_user_weights

    stratum_n = stratum_pop.get(bucket, global_stratum_pop)
    content_n = norm_user_scores(b2p_content_scores(uid))
    mf_n = norm_user_scores(b2p_mf_scores(uid))

    final_score = (
        weights["stratum_pop"] * stratum_n +
        weights["content_match"] * content_n +
        weights["bayes_score"] * b2p_bayes_n +
        weights["global_pop"] * b2p_pop_n +
        weights["cold_start"] * b2p_cold_n +
        weights["low_count"] * b2p_low_count_n +
        weights["mf_score"] * mf_n
    )

    for item in seen:
        pos = candidate_pos_b2p.get(item)
        if pos is not None:
            final_score[pos] = -np.inf

    take_n = min(topn * 4, len(candidate_items_b2p))
    best_pos = np.argpartition(-final_score, take_n - 1)[:take_n]
    best_pos = best_pos[np.argsort(-final_score[best_pos])]

    recs = []
    used = set()
    for pos in best_pos:
        item = candidate_items_b2p[pos]
        if item in used or item in seen:
            continue
        recs.append(item)
        used.add(item)
        if len(recs) == topn:
            break

    return recs


def b2p_recs_for_users(user_list, topn=B2P_TOPN):
    rec_map = {}
    for n, uid in enumerate(user_list, start=1):
        rec_map[uid] = b2p_recs_for_user(uid, topn=topn)
        if n % 1000 == 0:
            print("B2P users finished:", n)
    return rec_map


val_users_b2p = sorted(val_reviews["user_id"].unique())
test_users_b2p = sorted(test_reviews["user_id"].unique())

print("making B2P validation recs...")
b2p_val_recs = b2p_recs_for_users(val_users_b2p, topn=B2P_TOPN)

print("making B2P test recs...")
b2p_test_recs = b2p_recs_for_users(test_users_b2p, topn=B2P_TOPN)

b2p_val_scores = evaluate_model(
    "b2p_personalized_popularity",
    b2p_val_recs,
    val_reviews,
    split_name="val",
    k_list=[5, 10, 20]
)

b2p_test_scores = evaluate_model(
    "b2p_personalized_popularity",
    b2p_test_recs,
    test_reviews,
    split_name="test",
    k_list=[5, 10, 20]
)

b2p_scores = pd.concat([b2p_val_scores, b2p_test_scores], ignore_index=True)
b2p_scores.to_csv(os.path.join(RESULTS_DIR, "b2p_personalized_popularity_scores.csv"), index=False)
b2p_weight_table.to_csv(os.path.join(RESULTS_DIR, "b2p_personalized_popularity_weights.csv"), index=False)

print("saved:", os.path.join(RESULTS_DIR, "b2p_personalized_popularity_scores.csv"))
print("B2P weights:")
display(b2p_weight_table)
display(b2p_scores)


B2P user-history buckets: ['n_1', 'n_11_20', 'n_21_plus', 'n_2_3', 'n_4_5', 'n_6_10']
B2P content matrix: (2890, 12000)
MF variables not found, training B2P SVD warm-user component.
making B2P validation recs...
B2P users finished: 1000
B2P users finished: 2000
B2P users finished: 3000
B2P users finished: 4000
B2P users finished: 5000
B2P users finished: 6000
B2P users finished: 7000
B2P users finished: 8000
B2P users finished: 9000
making B2P test recs...
B2P users finished: 1000
B2P users finished: 2000
B2P users finished: 3000
B2P users finished: 4000
B2P users finished: 5000
B2P users finished: 6000
saved: /content/yelp_philly_processed/model_results/b2p_personalized_popularity_scores.csv
B2P weights:


,user_group,signal,weight
0,cold_or_sparse_user,stratum_pop,0.35
1,cold_or_sparse_user,content_match,0.30
2,cold_or_sparse_user,bayes_score,0.15
3,cold_or_sparse_user,global_pop,0.05
4,cold_or_sparse_user,cold_start,0.10
5,cold_or_sparse_user,low_count,0.05
6,cold_or_sparse_user,mf_score,0.00
7,warm_user,stratum_pop,0.15
8,warm_user,content_match,0.15
9,warm_user,bayes_score,0.10


,model,split,group,k,users_eval,hit@k,recall@k,mrr@k,ndcg@k
0,b2p_personalized_popularity,val,all_positive,5,7551,0.066614,0.027935,0.034843,0.024823
1,b2p_personalized_popularity,val,cold_start_positive,5,363,0.000000,0.000000,0.000000,0.000000
2,b2p_personalized_popularity,val,all_positive,10,7551,0.107668,0.048403,0.040174,0.031534
3,b2p_personalized_popularity,val,cold_start_positive,10,363,0.000000,0.000000,0.000000,0.000000
4,b2p_personalized_popularity,val,all_positive,20,7551,0.160773,0.077743,0.043717,0.040026
5,b2p_personalized_popularity,val,cold_start_positive,20,363,0.000000,0.000000,0.000000,0.000000
6,b2p_personalized_popularity,test,all_positive,5,5003,0.058365,0.029640,0.031464,0.024327
7,b2p_personalized_popularity,test,cold_start_positive,5,183,0.000000,0.000000,0.000000,0.000000
8,b2p_personalized_popularity,test,all_positive,10,5003,0.096942,0.050398,0.036383,0.031262
9,b2p_personalized_popularity,test,cold_start_positive,10,183,0.000000,0.000000,0.000000,0.000000


In [ ]:
# # FIDR weight tuning cell
# # =========================================================
# # Tune the FIDR-inspired hybrid weights on VALIDATION only.
# # This does not rerun data cleaning or the temporal split.
# # Run after: fast reload -> evaluator -> FIDR hybrid cell.
# # =========================================================
# import math
# import numpy as np
# import pandas as pd

# needed_for_tuning = [
#     "user_factors", "item_factors", "user_to_row", "candidate_cols",
#     "candidate_items", "candidate_pos", "item_content_mat", "liked_map",
#     "bayes_n", "pop_n", "cold_n", "low_count_n",
#     "val_reviews", "train_reviews", "make_seen_map", "make_truth_map"
# ]
# missing = [name for name in needed_for_tuning if name not in globals()]
# if missing:
#     raise RuntimeError("Run the FIDR hybrid cell before this tuning cell. Missing: " + str(missing))

# TUNE_TOPK = 20
# MF_VAL_RECALL10 = 0.0291
# MF_VAL_NDCG10 = 0.0182

# val_truth = make_truth_map(val_reviews, cold_only=False)
# val_cold_truth = make_truth_map(val_reviews, cold_only=True)
# tune_users = sorted(val_truth.keys())
# tune_user_pos = {uid: i for i, uid in enumerate(tune_users)}
# print("validation users for tuning:", len(tune_users))
# print("cold-start validation users:", len(val_cold_truth))

# # Seen item indices, so recommendations do not include already-reviewed businesses.
# seen_map_tune = make_seen_map(train_reviews)
# seen_idx = []
# for uid in tune_users:
#     seen_idx.append([candidate_pos[x] for x in seen_map_tune.get(uid, set()) if x in candidate_pos])

# # Convert truth sets from business ids to candidate indices.
# def truth_to_idx(truth):
#     return {
#         uid: {candidate_pos[x] for x in items if x in candidate_pos}
#         for uid, items in truth.items()
#         if uid in tune_user_pos
#     }

# val_truth_idx = truth_to_idx(val_truth)
# val_cold_truth_idx = truth_to_idx(val_cold_truth)


# def row_minmax(mat):
#     mat = np.asarray(mat, dtype=np.float32)
#     lo = mat.min(axis=1, keepdims=True)
#     hi = mat.max(axis=1, keepdims=True)
#     denom = np.where(hi > lo, hi - lo, 1.0)
#     return (mat - lo) / denom


# print("precomputing MF scores once...")
# val_rows = np.array([user_to_row[uid] for uid in tune_users], dtype=np.int64)
# mf_tune = user_factors[val_rows] @ item_factors.T
# mf_tune = row_minmax(mf_tune[:, candidate_cols])

# print("precomputing content scores once...")
# content_rows = []
# for uid in tune_users:
#     liked_items = liked_map.get(uid, [])
#     item_rows = [candidate_pos[item] for item in liked_items if item in candidate_pos]
#     if not item_rows:
#         content_rows.append(np.zeros(len(candidate_items), dtype=np.float32))
#     else:
#         user_profile = item_content_mat[item_rows].mean(axis=0)
#         scores = user_profile @ item_content_mat.T
#         content_rows.append(np.asarray(scores).ravel().astype(np.float32))
# content_tune = row_minmax(np.vstack(content_rows))
# print("score pieces ready:", mf_tune.shape, content_tune.shape)


# def metrics_from_top(top_idx, truth_idx, k=10):
#     hits_out, recall_out, rr_out, ndcg_out = [], [], [], []

#     for uid, true_items in truth_idx.items():
#         if len(true_items) == 0:
#             continue

#         u = tune_user_pos[uid]
#         recs = top_idx[u, :k]
#         flags = [1 if int(item) in true_items else 0 for item in recs]
#         hits = sum(flags)

#         hits_out.append(1.0 if hits else 0.0)
#         recall_out.append(hits / len(true_items))

#         rr = 0.0
#         for rank, hit in enumerate(flags, start=1):
#             if hit:
#                 rr = 1.0 / rank
#                 break
#         rr_out.append(rr)

#         dcg = 0.0
#         for rank, hit in enumerate(flags, start=1):
#             if hit:
#                 dcg += 1.0 / math.log2(rank + 1)

#         ideal_hits = min(len(true_items), k)
#         idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
#         ndcg_out.append(0.0 if idcg == 0 else dcg / idcg)

#     return {
#         "hit": float(np.mean(hits_out)) if hits_out else 0.0,
#         "recall": float(np.mean(recall_out)) if recall_out else 0.0,
#         "mrr": float(np.mean(rr_out)) if rr_out else 0.0,
#         "ndcg": float(np.mean(ndcg_out)) if ndcg_out else 0.0,
#     }


# def top_items_for_weights(w):
#     score = (
#         w["mf_score"] * mf_tune +
#         w["content_score"] * content_tune +
#         w["bayes_score"] * bayes_n[None, :] +
#         w["popularity"] * pop_n[None, :] +
#         w["cold_start"] * cold_n[None, :] +
#         w["low_count"] * low_count_n[None, :]
#     ).astype(np.float32)

#     for u, idxs in enumerate(seen_idx):
#         if idxs:
#             score[u, idxs] = -np.inf

#     rough_top = np.argpartition(-score, TUNE_TOPK - 1, axis=1)[:, :TUNE_TOPK]
#     rough_scores = np.take_along_axis(score, rough_top, axis=1)
#     order = np.argsort(-rough_scores, axis=1)
#     return np.take_along_axis(rough_top, order, axis=1)


# # Focused grid: simple weights that sum to 1 and are easy to explain in the paper.
# weight_list = []
# for mf_w in [0.32, 0.35, 0.38, 0.40, 0.42, 0.45]:
#     for content_w in [0.22, 0.25, 0.27, 0.30]:
#         for bayes_w in [0.10, 0.13, 0.15]:
#             for pop_w in [0.05, 0.07, 0.10]:
#                 for cold_w in [0.04, 0.06, 0.08, 0.10, 0.12]:
#                     low_w = 1.0 - (mf_w + content_w + bayes_w + pop_w + cold_w)
#                     if 0.02 <= low_w <= 0.10:
#                         weight_list.append({
#                             "mf_score": round(mf_w, 2),
#                             "content_score": round(content_w, 2),
#                             "bayes_score": round(bayes_w, 2),
#                             "popularity": round(pop_w, 2),
#                             "cold_start": round(cold_w, 2),
#                             "low_count": round(low_w, 2),
#                         })

# # include the two settings we already tried
# weight_list += [
#     {"mf_score": 0.30, "content_score": 0.30, "bayes_score": 0.15, "popularity": 0.10, "cold_start": 0.10, "low_count": 0.05},
#     {"mf_score": 0.45, "content_score": 0.25, "bayes_score": 0.15, "popularity": 0.10, "cold_start": 0.03, "low_count": 0.02},
# ]

# # unique settings
# unique = {}
# for w in weight_list:
#     key = tuple(w[k] for k in ["mf_score", "content_score", "bayes_score", "popularity", "cold_start", "low_count"])
#     unique[key] = w
# weight_list = list(unique.values())
# print("weight settings to test:", len(weight_list))

# rows = []
# for n, w in enumerate(weight_list, start=1):
#     top_idx = top_items_for_weights(w)
#     all10 = metrics_from_top(top_idx, val_truth_idx, k=10)
#     cold10 = metrics_from_top(top_idx, val_cold_truth_idx, k=10)
#     all20 = metrics_from_top(top_idx, val_truth_idx, k=20)
#     cold20 = metrics_from_top(top_idx, val_cold_truth_idx, k=20)

#     row = dict(w)
#     row.update({
#         "all_recall10": all10["recall"],
#         "all_ndcg10": all10["ndcg"],
#         "cold_recall10": cold10["recall"],
#         "cold_ndcg10": cold10["ndcg"],
#         "all_recall20": all20["recall"],
#         "all_ndcg20": all20["ndcg"],
#         "cold_recall20": cold20["recall"],
#         "cold_ndcg20": cold20["ndcg"],
#     })
#     rows.append(row)

#     if n % 25 == 0 or n == len(weight_list):
#         print("tested", n, "of", len(weight_list))

# fidr_tuning_results = pd.DataFrame(rows)
# fidr_tuning_results["balanced_score"] = (
#     0.45 * fidr_tuning_results["all_ndcg10"] +
#     0.35 * fidr_tuning_results["cold_ndcg10"] +
#     0.10 * fidr_tuning_results["all_recall10"] +
#     0.10 * fidr_tuning_results["cold_recall10"]
# )

# fidr_tuning_results["mf_safe"] = (
#     (fidr_tuning_results["all_recall10"] >= MF_VAL_RECALL10) &
#     (fidr_tuning_results["all_ndcg10"] >= MF_VAL_NDCG10) &
#     (fidr_tuning_results["cold_recall10"] > 0)
# )

# best_all = fidr_tuning_results.sort_values(["all_ndcg10", "all_recall10"], ascending=False).head(5)
# best_cold = fidr_tuning_results[fidr_tuning_results["cold_recall10"] > 0].sort_values(
#     ["cold_ndcg10", "cold_recall10"], ascending=False
# ).head(5)
# best_balanced = fidr_tuning_results[fidr_tuning_results["cold_recall10"] > 0].sort_values(
#     "balanced_score", ascending=False
# ).head(10)
# best_mf_safe = fidr_tuning_results[fidr_tuning_results["mf_safe"]].sort_values(
#     "balanced_score", ascending=False
# ).head(10)

# if len(best_mf_safe) > 0:
#     chosen = best_mf_safe.iloc[0]
#     chosen_reason = "best balanced setting that still beats the MF validation Recall@10 and NDCG@10 thresholds"
# else:
#     chosen = best_balanced.iloc[0]
#     chosen_reason = "best balanced setting with nonzero cold-start Recall@10"

# best_hybrid_weights = {
#     "mf_score": float(chosen["mf_score"]),
#     "content_score": float(chosen["content_score"]),
#     "bayes_score": float(chosen["bayes_score"]),
#     "popularity": float(chosen["popularity"]),
#     "cold_start": float(chosen["cold_start"]),
#     "low_count": float(chosen["low_count"]),
# }

# print("\nChosen reason:", chosen_reason)
# print("best_hybrid_weights =")
# print(best_hybrid_weights)
# print("\nBest all-positive settings:")
# display(best_all)
# print("\nBest cold-start settings:")
# display(best_cold)
# print("\nBest balanced settings:")
# display(best_balanced)
# print("\nBest MF-safe balanced settings:")
# display(best_mf_safe)

# # Save the tuning table for the paper / teammates.
# if "RESULTS_DIR" in globals():
#     tune_path = os.path.join(RESULTS_DIR, "fidr_weight_tuning_val.csv")
# else:
#     tune_path = os.path.join(DATA_DIR, "model_results", "fidr_weight_tuning_val.csv")
# fidr_tuning_results.sort_values("balanced_score", ascending=False).to_csv(tune_path, index=False)
# print("saved tuning results:", tune_path)


validation users for tuning: 7551
cold-start validation users: 363
precomputing MF scores once...
precomputing content scores once...
score pieces ready: (7551, 2890) (7551, 2890)
weight settings to test: 506
tested 25 of 506
tested 50 of 506
tested 75 of 506
tested 100 of 506
tested 125 of 506
tested 150 of 506
tested 175 of 506
tested 200 of 506
tested 225 of 506
tested 250 of 506
tested 275 of 506
tested 300 of 506
tested 325 of 506
tested 350 of 506
tested 375 of 506
tested 400 of 506
tested 425 of 506
tested 450 of 506
tested 475 of 506
tested 500 of 506
tested 506 of 506

Chosen reason: best balanced setting that still beats the MF validation Recall@10 and NDCG@10 thresholds
best_hybrid_weights =
{'mf_score': 0.45, 'content_score': 0.27, 'bayes_score': 0.1, 'popularity': 0.1, 'cold_start': 0.06, 'low_count': 0.02}

Best all-positive settings:


,mf_score,content_score,bayes_score,popularity,cold_start,low_count,all_recall10,all_ndcg10,cold_recall10,cold_ndcg10,all_recall20,all_ndcg20,cold_recall20,cold_ndcg20,balanced_score,mf_safe
383,0.42,0.25,0.15,0.1,0.06,0.02,0.035371,0.022630,0.0,0.0,0.060638,0.029836,0.002755,0.000637,0.013721,False
456,0.45,0.22,0.15,0.1,0.06,0.02,0.035393,0.022617,0.0,0.0,0.061481,0.030137,0.002755,0.000637,0.013717,False
302,0.40,0.27,0.15,0.1,0.06,0.02,0.035123,0.022520,0.0,0.0,0.059381,0.029435,0.002755,0.000637,0.013646,False
475,0.45,0.25,0.13,0.1,0.04,0.03,0.035279,0.022482,0.0,0.0,0.060276,0.029531,0.000000,0.000000,0.013645,False
505,0.45,0.25,0.15,0.1,0.03,0.02,0.034789,0.022462,0.0,0.0,0.061596,0.030120,0.000000,0.000000,0.013587,False



Best cold-start settings:


,mf_score,content_score,bayes_score,popularity,cold_start,low_count,all_recall10,all_ndcg10,cold_recall10,cold_ndcg10,all_recall20,all_ndcg20,cold_recall20,cold_ndcg20,balanced_score,mf_safe
12,0.32,0.27,0.15,0.05,0.12,0.09,0.009008,0.006296,0.189394,0.078945,0.013111,0.007431,0.283287,0.103406,0.050304,False
47,0.35,0.25,0.13,0.05,0.12,0.10,0.009298,0.006383,0.186639,0.075790,0.013791,0.007557,0.291322,0.102892,0.048993,False
23,0.32,0.30,0.13,0.05,0.12,0.08,0.009733,0.006661,0.185262,0.074988,0.015085,0.008067,0.273186,0.097300,0.048742,False
19,0.32,0.30,0.10,0.07,0.12,0.09,0.010030,0.006782,0.182507,0.073563,0.015046,0.008160,0.258724,0.092860,0.048052,False
8,0.32,0.27,0.13,0.07,0.12,0.09,0.010642,0.007104,0.179752,0.073266,0.015327,0.008376,0.270432,0.096603,0.047880,False



Best balanced settings:


,mf_score,content_score,bayes_score,popularity,cold_start,low_count,all_recall10,all_ndcg10,cold_recall10,cold_ndcg10,all_recall20,all_ndcg20,cold_recall20,cold_ndcg20,balanced_score,mf_safe
12,0.32,0.27,0.15,0.05,0.12,0.09,0.009008,0.006296,0.189394,0.078945,0.013111,0.007431,0.283287,0.103406,0.050304,False
47,0.35,0.25,0.13,0.05,0.12,0.10,0.009298,0.006383,0.186639,0.075790,0.013791,0.007557,0.291322,0.102892,0.048993,False
23,0.32,0.30,0.13,0.05,0.12,0.08,0.009733,0.006661,0.185262,0.074988,0.015085,0.008067,0.273186,0.097300,0.048742,False
19,0.32,0.30,0.10,0.07,0.12,0.09,0.010030,0.006782,0.182507,0.073563,0.015046,0.008160,0.258724,0.092860,0.048052,False
66,0.35,0.27,0.13,0.05,0.12,0.08,0.010866,0.007289,0.182507,0.072197,0.015671,0.008580,0.275941,0.095954,0.047887,False
8,0.32,0.27,0.13,0.07,0.12,0.09,0.010642,0.007104,0.179752,0.073266,0.015327,0.008376,0.270432,0.096603,0.047880,False
3,0.32,0.25,0.15,0.07,0.12,0.09,0.010739,0.007140,0.179752,0.072962,0.015214,0.008362,0.280533,0.099326,0.047799,False
0,0.32,0.22,0.15,0.10,0.12,0.09,0.013203,0.008600,0.176997,0.070931,0.018430,0.010016,0.269513,0.094634,0.047716,False
85,0.35,0.30,0.10,0.05,0.12,0.08,0.010754,0.007159,0.181129,0.072000,0.015241,0.008399,0.262167,0.092552,0.047610,False
130,0.38,0.25,0.10,0.05,0.12,0.10,0.010569,0.006995,0.181129,0.070828,0.014446,0.008023,0.280073,0.096429,0.047108,False



Best MF-safe balanced settings:


,mf_score,content_score,bayes_score,popularity,cold_start,low_count,all_recall10,all_ndcg10,cold_recall10,cold_ndcg10,all_recall20,all_ndcg20,cold_recall20,cold_ndcg20,balanced_score,mf_safe
489,0.45,0.27,0.10,0.1,0.06,0.02,0.034539,0.022185,0.002755,0.000829,0.058796,0.029197,0.002755,0.000829,0.014003,True
271,0.40,0.25,0.15,0.1,0.08,0.02,0.032391,0.021134,0.004132,0.002271,0.053759,0.027271,0.015152,0.004921,0.013957,True
354,0.42,0.22,0.15,0.1,0.08,0.03,0.029362,0.018986,0.009642,0.004316,0.049907,0.024960,0.028926,0.009069,0.013955,True
417,0.42,0.30,0.10,0.1,0.06,0.02,0.034091,0.021892,0.002755,0.000829,0.057377,0.028576,0.002755,0.000829,0.013826,True
447,0.45,0.22,0.13,0.1,0.08,0.02,0.032587,0.020896,0.004132,0.001979,0.053746,0.027047,0.017906,0.005346,0.013768,True
446,0.45,0.22,0.13,0.1,0.06,0.04,0.032552,0.021260,0.002755,0.001738,0.054162,0.027523,0.004132,0.002129,0.013706,True
187,0.38,0.27,0.15,0.1,0.08,0.02,0.032502,0.021073,0.002755,0.001738,0.053250,0.027008,0.015152,0.004854,0.013617,True
292,0.40,0.27,0.13,0.1,0.06,0.04,0.031720,0.020799,0.002755,0.001738,0.052499,0.026761,0.005510,0.002387,0.013415,True
373,0.42,0.25,0.13,0.1,0.06,0.04,0.031580,0.020802,0.002755,0.001738,0.053014,0.026961,0.002755,0.001738,0.013403,True
270,0.40,0.25,0.15,0.1,0.06,0.04,0.031945,0.021253,0.002755,0.000981,0.053274,0.027319,0.005510,0.001630,0.013377,True


saved tuning results: /content/yelp_philly_processed/model_results/fidr_weight_tuning_val.csv


In [ ]:
# =========================================================
# LLM-Based Recommender — Implementation of Sanner et al. (2023)
# "Large Language Models are Competitive Near Cold-start
#  Recommenders for Language- and Item-based Preferences"
# =========================================================
# Paper method: score each candidate item using the log-likelihood
# that an LLM assigns to the item name given a user-preference prefix.
# Three prompt variants (Completion, Zero-shot, Few-shot) × three
# preference types (Items-only, Language-only, Language+Items).
#
# Our adaptation for Yelp restaurants:
#   - "Items" = restaurant names the user positively reviewed in train
#   - "Language" = a summary of those reviews (categories + star signal)
#     constructed from structured metadata (no free-text reviews used,
#     so the signal is fully reproducible)
#   - Scoring: because we cannot extract token log-likelihoods from the
#     Claude API, we ask the model to return a ranked JSON list from a
#     small candidate pool (~40 items).  This is equivalent in spirit to
#     the paper's scoring approach — the model sees the same preference
#     context and orders items from most to least preferred.
#   - We evaluate on a random sample of users (LLM_MAX_USERS) to keep
#     API cost manageable; set LLM_MAX_USERS = None to run on all users.
#   - Prompt variants implemented:
#       * Zero-shot  Items-only
#       * Zero-shot  Language-only
#       * Zero-shot  Language+Items
#       * Few-shot   Items-only        (k = LLM_FEW_SHOT_K examples)
#       * Few-shot   Language-only
#       * Few-shot   Language+Items
# =========================================================

import os, json, time, math, random
import numpy as np
import pandas as pd

# ── dependencies ──────────────────────────────────────────────────────────────
try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "q", "install", "requests"])
    import requests

# ── config ────────────────────────────────────────────────────────────────────
LLM_TOPN          = 100        # final rec-list length per user
LLM_CANDIDATE_N   = 40         # candidates shown to LLM per call (paper uses 40)
LLM_MAX_USERS     = 200        # set to None to run on ALL users (expensive)
LLM_FEW_SHOT_K    = 3          # number of few-shot examples (paper: k=3 best)
LLM_MAX_LIKED     = 5          # max liked items shown per user (paper uses 5)
LLM_RETRY_WAIT    = 2          # seconds between retries on rate-limit
LLM_MAX_RETRIES   = 3
LLM_RANDOM_SEED   = 172
RESULTS_DIR       = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

rng_llm = random.Random(LLM_RANDOM_SEED)

MIN_POSITIVE_STARS_LLM = MIN_POSITIVE_STARS  # from evaluator cell (default 4.0)

# ── guard: require evaluated context ─────────────────────────────────────────
needed = ["train_reviews", "val_reviews", "test_reviews",
          "businesses", "cold_start_business_ids",
          "make_seen_map", "evaluate_model"]
missing = [n for n in needed if n not in globals()]
if missing:
    raise RuntimeError(
        "Run the 'Fast reload' and 'Evaluator' cells first. Missing: " + str(missing)
    )

# ── build lookup tables ───────────────────────────────────────────────────────
candidate_items_llm = sorted(train_reviews["business_id"].unique())
candidate_set_llm   = set(candidate_items_llm)
seen_map_llm        = make_seen_map(train_reviews)

bid_to_name = (
    businesses.set_index("business_id")["name"]
    .to_dict()
)
bid_to_cats = (
    businesses.set_index("business_id")["categories"]
    .fillna("")
    .to_dict()
)

train_pop_llm = (
    train_reviews.groupby("business_id")
    .size()
    .sort_values(ascending=False)
    .index.tolist()
)

# positive history per user
train_pos_llm = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS_LLM].copy()
user_liked_map = (
    train_pos_llm.groupby("user_id")["business_id"]
    .apply(list)
    .to_dict()
)

# ── preference representation helpers ────────────────────────────────────────

def liked_item_names(uid, k=LLM_MAX_LIKED):
    """Return up to k restaurant names the user positively reviewed."""
    liked = user_liked_map.get(uid, [])
    # stable order: most-reviewed first (proxy for strongest signal)
    return [bid_to_name.get(b, b) for b in liked[:k] if bid_to_name.get(b)]


def language_description(uid, k=LLM_MAX_LIKED):
    """
    Build a short NL preference description from the user's liked items.
    Paper: users write free text; here we construct it from categories,
    mirroring the paper's structured-to-language conversion idea.
    """
    liked = user_liked_map.get(uid, [])[:k]
    if not liked:
        return "I enjoy a variety of restaurants."

    cat_counter: dict = {}
    for bid in liked:
        cats = bid_to_cats.get(bid, "")
        for c in [x.strip() for x in cats.split(",") if x.strip()]:
            cat_counter[c] = cat_counter.get(c, 0) + 1

    # top categories by frequency
    top_cats = sorted(cat_counter, key=lambda c: -cat_counter[c])[:6]
    names    = [bid_to_name.get(b, b) for b in liked if bid_to_name.get(b)]

    if top_cats:
        cat_str = ", ".join(top_cats[:4])
        return (
            f"I enjoy restaurants that serve {cat_str}. "
            f"Examples of places I liked include: {', '.join(names[:3])}."
        )
    return f"I enjoy places like {', '.join(names[:3])}."


# ── few-shot example pool ─────────────────────────────────────────────────────

def few_shot_examples(target_uid, k=LLM_FEW_SHOT_K):
    """
    Return k (desc, liked_names, one_extra_item) tuples from OTHER users.
    Mirrors paper's few-shot template: each example shows a user's preferences
    and one additional item they liked (the 'next movie preference').
    """
    pool = [u for u in user_liked_map if u != target_uid
            and len(user_liked_map[u]) >= 2]
    rng_llm.shuffle(pool)
    examples = []
    for uid in pool[:k * 3]:          # oversample, take first k valid
        liked = user_liked_map[uid]
        names = [bid_to_name.get(b, b) for b in liked if bid_to_name.get(b)]
        if len(names) < 2:
            continue
        desc  = language_description(uid)
        shown = names[:LLM_MAX_LIKED - 1]   # first 4 (paper uses item 5 as target)
        extra = names[LLM_MAX_LIKED - 1]    # 5th item is the "additional preference"
        examples.append((desc, shown, extra))
        if len(examples) == k:
            break
    return examples


# ── prompt builders ───────────────────────────────────────────────────────────

def _candidate_block(names):
    """Numbered list of candidate restaurant names."""
    return "\n".join(f"  {i+1}. {n}" for i, n in enumerate(names))


def build_zero_shot_items_prompt(uid, cand_names):
    liked = liked_item_names(uid)
    liked_str = ", ".join(liked) if liked else "no restaurants yet"
    return (
        f"I like the following restaurants: {liked_str}.\n\n"
        f"From the list below, rank ALL restaurants from most to least likely "
        f"that I would enjoy. Return ONLY a JSON array of the restaurant names "
        f"in ranked order, with no extra text.\n\n"
        f"Candidates:\n{_candidate_block(cand_names)}"
    )


def build_zero_shot_language_prompt(uid, cand_names):
    desc = language_description(uid)
    return (
        f"I describe the restaurants I like as follows: {desc}\n\n"
        f"From the list below, rank ALL restaurants from most to least likely "
        f"that I would enjoy. Return ONLY a JSON array of the restaurant names "
        f"in ranked order, with no extra text.\n\n"
        f"Candidates:\n{_candidate_block(cand_names)}"
    )


def build_zero_shot_lang_items_prompt(uid, cand_names):
    desc  = language_description(uid)
    liked = liked_item_names(uid)
    liked_str = ", ".join(liked) if liked else "no restaurants yet"
    return (
        f"I describe the restaurants I like as follows: {desc} "
        f"I also like the following restaurants: {liked_str}.\n\n"
        f"From the list below, rank ALL restaurants from most to least likely "
        f"that I would enjoy. Return ONLY a JSON array of the restaurant names "
        f"in ranked order, with no extra text.\n\n"
        f"Candidates:\n{_candidate_block(cand_names)}"
    )


def _few_shot_block_items(examples):
    lines = []
    for desc, shown, extra in examples:
        lines.append(f"User Restaurant Preferences: {', '.join(shown)}")
        lines.append(f"Additional User Restaurant Preference: {extra}")
    return "\n".join(lines)


def _few_shot_block_lang(examples):
    lines = []
    for desc, shown, extra in examples:
        names = shown + [extra]
        lines.append(f"User Description: {desc}")
        lines.append(f"User Restaurant Preferences: {', '.join(names)}")
    return "\n".join(lines)


def _few_shot_block_lang_items(examples):
    lines = []
    for desc, shown, extra in examples:
        lines.append(f"User Description: {desc}")
        lines.append(f"User Restaurant Preferences: {', '.join(shown)}")
        lines.append(f"Additional User Restaurant Preference: {extra}")
    return "\n".join(lines)


def build_few_shot_items_prompt(uid, cand_names):
    examples  = few_shot_examples(uid)
    liked     = liked_item_names(uid)
    ex_block  = _few_shot_block_items(examples)
    return (
        f"{ex_block}\n"
        f"User Restaurant Preferences: {', '.join(liked)}\n\n"
        f"From the list below, rank ALL restaurants as the ADDITIONAL restaurant "
        f"this user is most to least likely to enjoy. Return ONLY a JSON array "
        f"of restaurant names in ranked order, with no extra text.\n\n"
        f"Candidates:\n{_candidate_block(cand_names)}"
    )


def build_few_shot_language_prompt(uid, cand_names):
    examples = few_shot_examples(uid)
    desc     = language_description(uid)
    ex_block = _few_shot_block_lang(examples)
    return (
        f"{ex_block}\n"
        f"User Description: {desc}\n\n"
        f"From the list below, rank ALL restaurants this user is most to least "
        f"likely to enjoy. Return ONLY a JSON array of restaurant names in ranked "
        f"order, with no extra text.\n\n"
        f"Candidates:\n{_candidate_block(cand_names)}"
    )


def build_few_shot_lang_items_prompt(uid, cand_names):
    examples = few_shot_examples(uid)
    desc     = language_description(uid)
    liked    = liked_item_names(uid)
    ex_block = _few_shot_block_lang_items(examples)
    return (
        f"{ex_block}\n"
        f"User Description: {desc}\n"
        f"User Restaurant Preferences: {', '.join(liked)}\n\n"
        f"From the list below, rank ALL restaurants as the ADDITIONAL restaurant "
        f"this user is most to least likely to enjoy. Return ONLY a JSON array "
        f"of restaurant names in ranked order, with no extra text.\n\n"
        f"Candidates:\n{_candidate_block(cand_names)}"
    )


PROMPT_BUILDERS = {
    "llm_zero_shot_items":       build_zero_shot_items_prompt,
    "llm_zero_shot_language":    build_zero_shot_language_prompt,
    "llm_zero_shot_lang_items":  build_zero_shot_lang_items_prompt,
    "llm_few_shot_items":        build_few_shot_items_prompt,
    "llm_few_shot_language":     build_few_shot_language_prompt,
    "llm_few_shot_lang_items":   build_few_shot_lang_items_prompt,
}

# ── Anthropic API call ────────────────────────────────────────────────────────

def call_claude(prompt_text, retries=LLM_MAX_RETRIES):
    """
    Call Claude claude-sonnet-4-20250514 and return the response text.
    The model is instructed to return a JSON array of ranked restaurant names.
    """
    for attempt in range(retries + 1):
        try:
            resp = requests.post(
                "https://api.anthropic.com/v1/messages",
                headers={"Content-Type": "application/json"},
                json={
                    "model": "claude-sonnet-4-20250514",
                    "max_tokens": 1000,
                    "system": (
                        "You are a restaurant recommendation assistant. "
                        "When asked to rank restaurants, return ONLY a valid JSON "
                        "array of restaurant name strings, ranked from most to least "
                        "recommended. No explanation, no markdown fences, no preamble."
                    ),
                    "messages": [{"role": "user", "content": prompt_text}],
                },
                timeout=60,
            )
            resp.raise_for_status()
            return resp.json()["content"][0]["text"].strip()
        except Exception as exc:
            if attempt < retries:
                time.sleep(LLM_RETRY_WAIT * (attempt + 1))
            else:
                return None
    return None


# ── parse LLM ranking → business IDs ─────────────────────────────────────────

def parse_ranked_names(raw_text, cand_names, cand_bids):
    """
    Parse the JSON array returned by the LLM into an ordered list of business IDs.
    Falls back to the original candidate order for unrecognised names.
    """
    if raw_text is None:
        return cand_bids[:]

    # strip markdown fences if present
    clean = raw_text.strip()
    for fence in ["```json", "```"]:
        if clean.startswith(fence):
            clean = clean[len(fence):]
        if clean.endswith("```"):
            clean = clean[:-3]
    clean = clean.strip()

    try:
        ranked = json.loads(clean)
    except json.JSONDecodeError:
        return cand_bids[:]

    if not isinstance(ranked, list):
        return cand_bids[:]

    # build name→bid mapping (case-insensitive)
    name_to_bid = {n.lower(): b for n, b in zip(cand_names, cand_bids)}

    ordered, seen_set = [], set()
    for name in ranked:
        key = str(name).lower().strip()
        # exact match first, then prefix / substring match
        bid = name_to_bid.get(key)
        if bid is None:
            for k, b in name_to_bid.items():
                if key in k or k in key:
                    bid = b
                    break
        if bid and bid not in seen_set:
            ordered.append(bid)
            seen_set.add(bid)

    # append any missed candidates at the end (preserves full coverage)
    for bid in cand_bids:
        if bid not in seen_set:
            ordered.append(bid)
            seen_set.add(bid)

    return ordered


# ── candidate sampling for a user ────────────────────────────────────────────

def sample_candidates_for_user(uid, n=LLM_CANDIDATE_N):
    """
    Sample n candidate restaurants the user has NOT seen in train.
    Mirrors the paper's 40-item evaluation pool:
      - half from cold-start pool (paper emphasis on near-cold-start)
      - half from popular warm items (coverage baseline)
    """
    seen   = seen_map_llm.get(uid, set())

    cold   = [b for b in candidate_items_llm
              if b in cold_start_business_ids and b not in seen]
    warm   = [b for b in train_pop_llm
              if b not in cold_start_business_ids and b not in seen]

    rng_llm.shuffle(cold)
    cold_n = min(n // 2, len(cold))
    warm_n = min(n - cold_n, len(warm))

    sampled = cold[:cold_n] + warm[:warm_n]
    rng_llm.shuffle(sampled)
    return sampled


# ── per-user recommendation for a single variant ─────────────────────────────

def llm_recs_for_user(uid, prompt_builder, topn=LLM_TOPN):
    """
    1. Sample LLM_CANDIDATE_N candidates
    2. Build the prompt
    3. Call Claude → get ranked list of those candidates
    4. Pad to topn with popularity fallback
    """
    seen       = seen_map_llm.get(uid, set())
    cand_bids  = sample_candidates_for_user(uid, n=LLM_CANDIDATE_N)
    cand_names = [bid_to_name.get(b, b) for b in cand_bids]

    prompt     = prompt_builder(uid, cand_names)
    raw        = call_claude(prompt)
    ranked_bids = parse_ranked_names(raw, cand_names, cand_bids)

    # pad to topn with popularity fallback (unseen, not already in ranked)
    used = set(ranked_bids) | seen
    for bid in train_pop_llm:
        if bid not in used and bid not in seen:
            ranked_bids.append(bid)
            used.add(bid)
        if len(ranked_bids) >= topn:
            break

    return ranked_bids[:topn]


# ── run all six variants ──────────────────────────────────────────────────────

def run_llm_variant(variant_name, prompt_builder, eval_users, split_df,
                    split_name, topn=LLM_TOPN):
    """Generate recs for all eval_users and evaluate."""
    rec_map = {}
    for i, uid in enumerate(eval_users):
        rec_map[uid] = llm_recs_for_user(uid, prompt_builder, topn=topn)
        if (i + 1) % 25 == 0 or (i + 1) == len(eval_users):
            print(f"  [{variant_name}] {split_name}: {i+1}/{len(eval_users)} users done")

    scores = evaluate_model(
        variant_name, rec_map, split_df,
        split_name=split_name, k_list=[5, 10, 20]
    )
    return scores, rec_map


# ── select user sample ────────────────────────────────────────────────────────

all_val_users  = sorted(val_reviews["user_id"].unique())
all_test_users = sorted(test_reviews["user_id"].unique())

if LLM_MAX_USERS is not None:
    rng_llm.shuffle(all_val_users)
    rng_llm.shuffle(all_test_users)
    eval_val_users  = all_val_users[:LLM_MAX_USERS]
    eval_test_users = all_test_users[:LLM_MAX_USERS]
else:
    eval_val_users  = all_val_users
    eval_test_users = all_test_users

print(f"LLM evaluation: {len(eval_val_users)} val users, "
      f"{len(eval_test_users)} test users")
print(f"Prompt variants: {list(PROMPT_BUILDERS.keys())}")
print(f"Candidates per call: {LLM_CANDIDATE_N} | Few-shot k: {LLM_FEW_SHOT_K}\n")

# ── main loop — validate then test each variant ───────────────────────────────

all_llm_scores = []

for variant_name, builder in PROMPT_BUILDERS.items():
    print(f"\n{'='*60}")
    print(f"  Variant: {variant_name}")
    print(f"{'='*60}")

    val_scores, _  = run_llm_variant(
        variant_name, builder,
        eval_val_users, val_reviews, split_name="val"
    )
    test_scores, _ = run_llm_variant(
        variant_name, builder,
        eval_test_users, test_reviews, split_name="test"
    )

    combined = pd.concat([val_scores, test_scores], ignore_index=True)
    all_llm_scores.append(combined)

    save_path = os.path.join(RESULTS_DIR, f"{variant_name}_scores.csv")
    combined.to_csv(save_path, index=False)
    print(f"  saved: {save_path}")
    display(combined[combined["k"] == 10])

# ── aggregate summary table ───────────────────────────────────────────────────

llm_all = pd.concat(all_llm_scores, ignore_index=True)
llm_all.to_csv(os.path.join(RESULTS_DIR, "llm_all_variants_scores.csv"), index=False)

print("\n" + "="*60)
print("LLM SUMMARY — all variants @ k=10, test split, all_positive group")
print("="*60)
summary = (
    llm_all[
        (llm_all["split"] == "test") &
        (llm_all["k"] == 10) &
        (llm_all["group"] == "all_positive")
    ][["model", "recall@k", "ndcg@k", "mrr@k", "hit@k", "users_eval"]]
    .sort_values("ndcg@k", ascending=False)
    .reset_index(drop=True)
)
display(summary)

print("\nLLM SUMMARY — cold-start users @ k=10, test split")
cold_summary = (
    llm_all[
        (llm_all["split"] == "test") &
        (llm_all["k"] == 10) &
        (llm_all["group"] == "cold_start_positive")
    ][["model", "recall@k", "ndcg@k", "mrr@k", "hit@k", "users_eval"]]
    .sort_values("ndcg@k", ascending=False)
    .reset_index(drop=True)
)
display(cold_summary)

print(f"\nAll results saved to: {RESULTS_DIR}")
print("Files written:")
for v in PROMPT_BUILDERS:
    print(f"  - {v}_scores.csv")
print("  - llm_all_variants_scores.csv")
